### Mesoscale Convective System (MCS) Tracking

In [6]:
import warnings; warnings.filterwarnings("ignore")

import os, re, glob, gc
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

from scipy import ndimage as ndi
from scipy.stats import gaussian_kde, linregress, pearsonr, spearmanr

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D

# ---------------- USER SETTINGS ----------------
FILE_GLOB = "/glade/work/sshohan/MPAS_MCS_Tracking/hourly_subset/regridded_to_MERGIR/regridded*.nc"
TB_VAR = "ctt"

# Precip overlay for snapshots
HAS_PRECIP = True
PRECIP_VAR = "prec_total"
PRECIP_UNITS = "mm h$^{-1}$"

# KDE overlay for OVERVIEW
PRECIP_KDE_THRESH = 20.0
MAX_PTS_PER_HOUR  = 12000
KDE_LEVELS        = [0.2, 0.4, 0.6, 0.8]  # normalized 0–1 contour levels
KDE_BW_ADJUST     = 1.00
# --- Snapshot start-marker settings ---
PLOT_START_ON_ALL_STEPS = True   # if False, only marks on step 1

# Smart placement controls (reduce overlaps with track points/text already placed)
LABEL_MIN_DIST_DEG   = 0.50
LABEL_MIN_DIST_TEXT  = 0.65
LABEL_MIN_EDGE_PAD   = 0.35
LABEL_TRY_RADII_DEG  = [0.55, 0.85, 1.15, 1.45, 1.80]

# label style
START_TEXT_COLOR = "deeppink"
START_TEXT_FS    = 14
START_ARROW_LW   = 1.8

# --- Snapshot: show full MCS object pixels (distinguishable from centroid) ---
SHOW_OBJECT_PIXELS_IN_SNAPSHOTS = True
OBJ_PIX_COLOR  = "magenta"   # object pixels fill
OBJ_PIX_ALPHA  = 0.22        # transparency
OBJ_EDGE_COLOR = "magenta"   # outline color
OBJ_EDGE_LW    = 2.0

# Provided precip colors/levels
PRECIP_COLOURS = ['white','#00EEEE','#00B2EE','#1E90FF','#1f77b4','#104E8B','#7FFF00',
                  '#00CD00',"#53a447",'#008B00','#FFFF00','#FFD700','#CD8500','#FF7F00',
                  '#EE4000','#CD0000','#8B0000','#8968CD','#912CEE','#8B008B']
PRECIP_LEVELS = [4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42]

TB_CCS_THRESH = 241.0
TB_CORE_THRESH = 225.0
MCS_MIN_AREA_KM2 = 60000.0
MCS_MIN_DURATION_H = 6

CONNECTIVITY = 2
MAX_LINK_DIST_KM = 250.0

DET_OUT = "mcs_detections.csv"
TRK_OUT = "mcs_tracks.csv"
FIG_DIR_OV = "figs_overview"
FIG_DIR_SNP = "figs_snapshots"
DET_TMP = "_tmp_detections_stream.csv"
# Snapshot map extent (must match make_map)
MAP_EXTENT = (-107.42619, -91.85856, 25.02549, 37.27808)  # (lonmin, lonmax, latmin, latmax)

CITIES = {
    'Dallas':  (-96.48, 32.46),
    'Houston': (-95.22, 29.45),
    'Austin':  (-97.44, 30.16),
}

# --- NEW: lifecycle diagnostics (no precip) ---
LIFECYCLE_NBINS = 20
INTENSE_Q = 0.90  # "severe proxy" threshold = top 10% intensity across all hourly samples

def parse_hour(fname):
    """
    Parse filenames like:
      regridded_MPAS_hourly_Texas_subset_20150504_00.nc

    Returns
    -------
    ts : pandas.Timestamp or None
        Timestamp for the file hour, e.g. 2015-05-04 00:00:00
    key : str or None
        Formatted key, e.g. '20150504-00'
    """
    base = os.path.basename(fname)

    m = re.search(r"regridded_MPAS_hourly_Texas_subset_(\d{8})_(\d{2})\.nc$", base)

    if m:
        date_str = m.group(1)   # YYYYMMDD
        hour_str = m.group(2)   # HH

        ts = pd.to_datetime(
            date_str + hour_str,
            format="%Y%m%d%H",
            errors="coerce"
        )

        if pd.isna(ts):
            return None, None

        return ts, ts.strftime("%Y%m%d-%H")

    return None, None

def extract_lat_lon_tb(ds, tb_var=TB_VAR, p_var=None):
    da = ds[tb_var].isel(time=0)
    dims = list(da.dims)
    lat_dim = next((d for d in dims if 'lat' in d.lower() or d.lower() == 'y'), None)
    lon_dim = next((d for d in dims if 'lon' in d.lower() or d.lower() == 'x'), None)
    if lat_dim is None or lon_dim is None:
        lat_dim = 'lat' if 'lat' in ds.coords else dims[-2]
        lon_dim = 'lon' if 'lon' in ds.coords else dims[-1]

    lat = ds[lat_dim].values
    lon = ds[lon_dim].values

    if lat.ndim == 2 and lon.ndim == 2:
        lat1d = lat[:, 0]
        lon1d = lon[0, :]
    else:
        lat1d = lat
        lon1d = lon

    Tb2d = da.transpose(lat_dim, lon_dim).astype(np.float32).values

    P2d = None
    if (p_var is not None) and (p_var in ds.variables):
        P2d = ds[p_var].isel(time=0).transpose(lat_dim, lon_dim).astype(np.float32).values

    return Tb2d, lat1d.astype(np.float32), lon1d.astype(np.float32), P2d

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    d = np.deg2rad
    lat1 = d(lat1); lon1 = d(lon1)
    lat2 = d(lat2); lon2 = d(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return 2.0*R*np.arcsin(np.sqrt(a))

def label_objects(Tb2d):
    mask = Tb2d < TB_CCS_THRESH
    lbl, num = ndi.label(mask, ndi.generate_binary_structure(2, CONNECTIVITY))
    return lbl, num

def compute_area(lat, lon):
    lat = np.asarray(lat, dtype=np.float32)
    lon = np.asarray(lon, dtype=np.float32)
    if lat.size < 2 or lon.size < 2:
        raise RuntimeError("[ERROR] compute_area requires at least 2 lat and 2 lon points.")
    dlat = float(np.abs(lat[1]-lat[0]))
    dlon = float(np.abs(lon[1]-lon[0]))
    km_per_deg_lat = 111.32
    km_per_deg_lon = 111.32 * np.cos(np.deg2rad(lat))
    cell_area_row = (km_per_deg_lat * dlat) * (km_per_deg_lon * dlon)   # (nlat,)
    return (cell_area_row[:, None] * np.ones((1, lon.size), dtype=np.float32)).astype(np.float32)

def add_latlon_ticks(ax, fontsize=14):
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.0)
    gl.xlines = False
    gl.ylines = False
    gl.top_labels = False
    gl.right_labels = False
    try:
        gl.xlabel_style = {"size": fontsize}
        gl.ylabel_style = {"size": fontsize}
    except Exception:
        pass

def make_map(title):
    proj = ccrs.PlateCarree()
    fig, ax = plt.subplots(figsize=(11.2, 9.0), subplot_kw=dict(projection=proj))
    ax.set_extent(list(MAP_EXTENT), crs=proj)
    ax.set_title(title, fontsize=16, weight="bold")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='k', zorder=50)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='k', zorder=51)
    try:
        ax.add_feature(cfeature.STATES.with_scale("10m"), linewidth=0.7, edgecolor='k',
                       facecolor='none', zorder=52)
    except Exception:
        ax.add_feature(cfeature.STATES, linewidth=0.7, edgecolor='k', facecolor='none', zorder=52)
    return fig, ax, proj

def write_chunk(df, path):
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=header, index=False)

def _kde_on_grid(lon_pts, lat_pts, lon_grid_1d, lat_grid_1d, bw_adjust=1.0):
    lon_pts = np.asarray(lon_pts, dtype=np.float64)
    lat_pts = np.asarray(lat_pts, dtype=np.float64)
    ok = np.isfinite(lon_pts) & np.isfinite(lat_pts)
    lon_pts = lon_pts[ok]; lat_pts = lat_pts[ok]
    if lon_pts.size < 10:
        return None

    values = np.vstack([lon_pts, lat_pts])
    kde = gaussian_kde(values)
    if bw_adjust != 1.0:
        kde.set_bandwidth(bw_method=kde.factor * bw_adjust)

    LON, LAT = np.meshgrid(lon_grid_1d.astype(np.float64), lat_grid_1d.astype(np.float64))
    positions = np.vstack([LON.ravel(), LAT.ravel()])
    Z = kde(positions).reshape(LAT.shape)

    zmax = np.nanmax(Z)
    if not np.isfinite(zmax) or zmax <= 0:
        return None
    return (Z / zmax).astype(np.float32)

# ---- NEW: safe contour levels helper (prevents contour() errors when Z has degenerate range) ----
def _safe_levels(Z, levels):
    if Z is None:
        return None
    zmin = float(np.nanmin(Z))
    zmax = float(np.nanmax(Z))
    if (not np.isfinite(zmin)) or (not np.isfinite(zmax)) or (zmax <= zmin):
        return None
    lv = [float(l) for l in levels if (float(l) > zmin) and (float(l) < zmax)]
    return lv if len(lv) else None

def choose_nonoverlap_label_anchor(start_lon, start_lat, track_lonlat, used_label_anchors,
                                  extent=MAP_EXTENT):
    lonmin, lonmax, latmin, latmax = extent
    directions = [
        ( 1,  1), (-1,  1), ( 1, -1), (-1, -1),
        ( 1,  0), (-1,  0), ( 0,  1), ( 0, -1),
    ]
    candidates = []
    for r in LABEL_TRY_RADII_DEG:
        for sx, sy in directions:
            candidates.append((sx*r, sy*r))
    candidates += [(2.3, 1.1), (-2.3, 1.1), (2.3, -1.1), (-2.3, -1.1)]

    track_lonlat = np.asarray(track_lonlat, dtype=np.float64)
    used_label_anchors = np.asarray(used_label_anchors, dtype=np.float64) if len(used_label_anchors) else None

    for dx, dy in candidates:
        tx = start_lon + dx
        ty = start_lat + dy
        if (tx < lonmin + LABEL_MIN_EDGE_PAD) or (tx > lonmax - LABEL_MIN_EDGE_PAD):
            continue
        if (ty < latmin + LABEL_MIN_EDGE_PAD) or (ty > latmax - LABEL_MIN_EDGE_PAD):
            continue
        if track_lonlat.size > 0:
            dtrk = np.sqrt((track_lonlat[:, 0] - tx)**2 + (track_lonlat[:, 1] - ty)**2)
            if np.nanmin(dtrk) < LABEL_MIN_DIST_DEG:
                continue
        if used_label_anchors is not None and used_label_anchors.size > 0:
            dlbl = np.sqrt((used_label_anchors[:, 0] - tx)**2 + (used_label_anchors[:, 1] - ty)**2)
            if np.nanmin(dlbl) < LABEL_MIN_DIST_TEXT:
                continue
        return tx, ty

    return start_lon - 0.9, start_lat + 0.9

def save_density_map(count2d, lon1d, lat1d, title, out_png, cmap="gist_ncar",
                     vmin=0.0, vmax=100.0, cbar_label="Density Grid Map"):
    fig, ax, pc = make_map(title)
    add_latlon_ticks(ax, fontsize=16)

    Z = np.clip(count2d.astype(np.float32), vmin, vmax)

    pm = ax.pcolormesh(lon1d, lat1d, Z, transform=pc, cmap=cmap, shading="auto",
                       vmin=vmin, vmax=vmax, zorder=2)

    for city, (clon, clat) in CITIES.items():
        ax.plot(clon, clat, 'kD', transform=pc, zorder=10)
        ax.text(clon - 0.5, clat - 0.5, city, transform=pc,
                fontsize=14, color='k', zorder=11)

    cb_ax = fig.add_axes([0.875, 0.25, 0.02, 0.50])
    cb = plt.colorbar(pm, cax=cb_ax)
    cb.set_label(cbar_label, fontsize=16)
    cb.ax.tick_params(labelsize=16)

    fig.subplots_adjust(left=0.06, right=0.865, top=0.93, bottom=0.12)
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

# ==========================
# UPDATED: grid intensity map + optional KDE contours overlay
# ==========================
def save_grid_intensity_map(mean_int2d, lon1d, lat1d, title, out_png,
                            cmap="gist_ncar", vmin=None, vmax=None,
                            cbar_label="Grid Mean Intensity (proxy)",
                            contour_Z=None, contour_levels=None,
                            contour_color="red", contour_ls="--", contour_lw=2.2,
                            contour_label=None):
    fig, ax, pc = make_map(title)
    add_latlon_ticks(ax, fontsize=16)

    Z = mean_int2d.astype(np.float32)

    if vmin is None:
        vmin = float(np.nanpercentile(Z[np.isfinite(Z)], 2)) if np.any(np.isfinite(Z)) else 0.0
    if vmax is None:
        vmax = float(np.nanpercentile(Z[np.isfinite(Z)], 98)) if np.any(np.isfinite(Z)) else 1.0
    if not np.isfinite(vmin): vmin = 0.0
    if not np.isfinite(vmax): vmax = 1.0
    if vmax <= vmin: vmax = vmin + 1e-6

    pm = ax.pcolormesh(lon1d, lat1d, Z, transform=pc, cmap=cmap, shading="auto",
                       vmin=vmin, vmax=vmax, zorder=2)

    legend_handles = []
    # ---- overlay KDE contours (heavy_Z is already normalized 0–1) ----
    if (contour_Z is not None) and (contour_levels is not None):
        lv = _safe_levels(contour_Z, contour_levels)
        if lv is not None:
            ax.contour(lon1d, lat1d, contour_Z,
                       levels=lv,
                       colors=contour_color,
                       linestyles=contour_ls,
                       linewidths=contour_lw,
                       transform=pc, zorder=8)
            if contour_label:
                legend_handles.append(
                    Line2D([0], [0], color=contour_color, lw=contour_lw, ls=contour_ls,
                           label=contour_label)
                )

    for city, (clon, clat) in CITIES.items():
        ax.plot(clon, clat, 'kD', transform=pc, zorder=10)
        ax.text(clon - 0.5, clat - 0.5, city, transform=pc,
                fontsize=14, color='k', zorder=11)

    if legend_handles:
        ax.legend(handles=legend_handles, loc="lower left",
                  fontsize=14, frameon=True, framealpha=0.95)

    cb_ax = fig.add_axes([0.875, 0.25, 0.02, 0.50])
    cb = plt.colorbar(pm, cax=cb_ax)
    cb.set_label(cbar_label, fontsize=16)
    cb.ax.tick_params(labelsize=16)

    fig.subplots_adjust(left=0.06, right=0.865, top=0.93, bottom=0.12)
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

# ---------------- STATS HELPERS ----------------
def _scatter_with_regression(x, y, xlabel, ylabel, title, out_png):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]; y = y[ok]
    if x.size < 3:
        return

    lr = linregress(x, y)
    r_pear, p_pear = pearsonr(x, y)
    r_spear, p_spear = spearmanr(x, y)

    fig, ax = plt.subplots(figsize=(8.2, 6.6))
    ax.scatter(x, y, s=55, edgecolor="k", linewidth=0.6, alpha=0.85)

    xx = np.linspace(np.nanmin(x), np.nanmax(x), 200)
    ax.plot(xx, lr.intercept + lr.slope * xx, linewidth=2.2)

    ax.set_xlabel(xlabel, fontsize=14)
    ax.set_ylabel(ylabel, fontsize=14)
    ax.set_title(title, fontsize=15, weight="bold")

    txt = (
        f"Linear fit: y = {lr.slope:.3g} x + {lr.intercept:.3g}\n"
        f"Pearson r = {r_pear:.3f} (p={p_pear:.2g})\n"
    )
    ax.text(0.02, 0.98, txt, transform=ax.transAxes,
            ha="left", va="top", fontsize=12,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.9, linewidth=0.8))

    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

def _histplot(x, xlabel, title, out_png, bins=18):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size < 3:
        return
    fig, ax = plt.subplots(figsize=(8.2, 6.2))
    ax.hist(x, bins=bins, edgecolor="k", linewidth=0.7, alpha=0.9)
    ax.set_xlabel(xlabel, fontsize=14)
    ax.set_ylabel("Count", fontsize=14)
    ax.set_title(title, fontsize=15, weight="bold")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

def scatter_grid_with_regression(x2d, y2d,
                                 xlabel, ylabel, title, out_png,
                                 min_points=30,
                                 stats_loc="topleft"):
    """
    Scatter plot between two gridded fields with regression + correlations

    stats_loc:
        "topleft"   → stats box at upper-left
        "lowerright"→ stats box at lower-right
    """
    x = np.asarray(x2d, dtype=np.float64).ravel()
    y = np.asarray(y2d, dtype=np.float64).ravel()

    ok = np.isfinite(x) & np.isfinite(y)
    ok = ok & (x >= 0) & (y >= 0)
    x = x[ok]
    y = y[ok]

    if x.size < min_points:
        print(f"[WARN] Not enough valid points for scatter: {out_png}")
        return

    # Correlations
    r_p, p_p = pearsonr(x, y)
    r_s, p_s = spearmanr(x, y)

    # Linear regression
    lr = linregress(x, y)
    xx = np.linspace(np.nanmin(x), np.nanmax(x), 200)
    yy = lr.intercept + lr.slope * xx
    yy = np.clip(yy, 0, None)

    fig, ax = plt.subplots(figsize=(8.5, 6.8))
    ax.scatter(x, y, s=28, alpha=0.55,
               edgecolor="k", linewidth=0.3)

    ax.plot(xx, yy,
        color="red", lw=2.4,
        label="Linear regression")

    ax.set_xlabel(xlabel, fontsize=15)
    ax.set_ylabel(ylabel, fontsize=15)
    ax.set_title(title, fontsize=16, weight="bold")

    stats_txt = (
        f"Pearson r = {r_p:.3f} (p={p_p:.2g})\n"
        f"Slope = {lr.slope:.3g}"
    )

    # ---------- Stats box placement ----------
    if stats_loc == "lowerright":
        ax.text(0.98, 0.18, stats_txt,
        transform=ax.transAxes,
        ha="right", va="bottom",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.22",
                  facecolor="white",
                  alpha=0.95,
                  linewidth=0.7))

    else:  # "topleft" (default)
        ax.text(0.02, 0.98, stats_txt,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.25",
                  facecolor="white",
                  alpha=0.95,
                  linewidth=0.7))


    ax.grid(True, alpha=0.25)

    # Legend always bottom-right
    ax.legend(loc="lower right", fontsize=13, frameon=True)

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)


def compute_mcs_lifecycle_stats(det_centroids):
    df = det_centroids.copy()
    df = df.sort_values(["track", "time"]).copy()

    out_rows = []
    for tid, g in df.groupby("track", sort=True):
        g = g.sort_values("time")
        if len(g) < 2:
            continue

        t0 = pd.to_datetime(g["time"].iloc[0])
        t1 = pd.to_datetime(g["time"].iloc[-1])
        dur_h = float((t1 - t0).total_seconds() / 3600.0) + 1.0

        lat = g["clat"].to_numpy(np.float64)
        lon = g["clon"].to_numpy(np.float64)
        times = pd.to_datetime(g["time"]).to_numpy()

        d_km = []
        dt_h = []
        for i in range(1, len(g)):
            dk = float(haversine(lat[i-1], lon[i-1], lat[i], lon[i]))
            dh = float((pd.to_datetime(times[i]) - pd.to_datetime(times[i-1])).total_seconds() / 3600.0)
            if np.isfinite(dk) and np.isfinite(dh) and dh > 0:
                d_km.append(dk)
                dt_h.append(dh)

        d_km = np.asarray(d_km, dtype=np.float64)
        dt_h = np.asarray(dt_h, dtype=np.float64)

        total_dist_km  = float(np.nansum(d_km)) if d_km.size else np.nan
        mean_speed_kmh = float(np.nanmean(d_km / dt_h)) if d_km.size else np.nan
        max_speed_kmh  = float(np.nanmax(d_km / dt_h))  if d_km.size else np.nan

        area  = g["area"].to_numpy(np.float64)
        tbmin = g["tbmin"].to_numpy(np.float64)
        inten = g["intensity"].to_numpy(np.float64)

        iA = int(np.nanargmax(area))
        iI = int(np.nanargmax(inten))
        iT = int(np.nanargmin(tbmin))

        age_h = (pd.to_datetime(g["time"]) - t0).dt.total_seconds().to_numpy(np.float64) / 3600.0
        okA = np.isfinite(age_h) & np.isfinite(area)
        if np.sum(okA) >= 3:
            lrA = linregress(age_h[okA], area[okA])
            area_slope_km2_per_h = float(lrA.slope)
        else:
            area_slope_km2_per_h = np.nan

        out_rows.append(dict(
            track=int(tid),
            start_time=t0,
            end_time=t1,
            duration_h=dur_h,
            n_hours=int(len(g)),
            max_area_km2=float(np.nanmax(area)),
            time_of_max_area=pd.to_datetime(g["time"].iloc[iA]),
            peak_intensity=float(np.nanmax(inten)),
            time_of_peak_intensity=pd.to_datetime(g["time"].iloc[iI]),
            min_tb_k=float(np.nanmin(tbmin)),
            time_of_min_tb=pd.to_datetime(g["time"].iloc[iT]),
            total_distance_km=total_dist_km,
            mean_speed_kmh=mean_speed_kmh,
            max_speed_kmh=max_speed_kmh,
            area_trend_km2_per_h=area_slope_km2_per_h
        ))

    return pd.DataFrame(out_rows)

# ---------------- lifecycle composite + peak timing + "intense probability" ----------------
def lifecycle_composite(df_centroids, n_bins=20,
                        time_col="time", track_col="track",
                        intensity_col="intensity",
                        tb_col="tbmin", area_col="area"):
    df = df_centroids.copy()
    df = df.sort_values([track_col, time_col]).copy()

    g = df.groupby(track_col)
    t0 = g[time_col].transform("min")
    t1 = g[time_col].transform("max")

    denom = (t1 - t0).dt.total_seconds().astype(np.float64)
    ok = denom > 0
    df = df.loc[ok].copy()
    denom = denom[ok]

    tau = (df[time_col] - t0[ok]).dt.total_seconds().astype(np.float64) / denom
    df["tau"] = np.clip(tau, 0.0, 1.0)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    df["tau_bin"] = np.digitize(df["tau"].to_numpy(np.float64), edges, right=True) - 1
    df = df[(df["tau_bin"] >= 0) & (df["tau_bin"] < n_bins)].copy()

    def _summ(x):
        x = np.asarray(x, dtype=np.float64)
        x = x[np.isfinite(x)]
        if x.size == 0:
            return dict(q25=np.nan, q50=np.nan, q75=np.nan, mean=np.nan, n=0)
        return dict(
            q25=float(np.nanpercentile(x, 25)),
            q50=float(np.nanpercentile(x, 50)),
            q75=float(np.nanpercentile(x, 75)),
            mean=float(np.nanmean(x)),
            n=int(x.size)
        )

    out = {}
    for vname, col in [("intensity", intensity_col), ("tbmin", tb_col), ("area", area_col)]:
        if col is None or col not in df.columns:
            continue
        stats = []
        for b in range(n_bins):
            stats.append(_summ(df.loc[df["tau_bin"] == b, col]))
        out[vname] = pd.DataFrame(stats)
    return centers, out, df

def plot_lifecycle_composites(tau, stats_dict, out_png,
                             title="MCS Lifecycle Composite (Tb/Area-derived proxies; no precip)",
                             show_intensity=True, show_tb=True, show_area=True):
    panels = []
    if show_intensity and "intensity" in stats_dict: panels.append("intensity")
    if show_tb and "tbmin" in stats_dict: panels.append("tbmin")
    if show_area and "area" in stats_dict: panels.append("area")
    if len(panels) == 0:
        return

    fig, axes = plt.subplots(len(panels), 1, figsize=(9.2, 3.2*len(panels)), sharex=True)
    if len(panels) == 1:
        axes = [axes]

    for ax, key in zip(axes, panels):
        st = stats_dict[key]
        ax.plot(tau, st["q50"].to_numpy(np.float64), linewidth=2.2)
        ax.fill_between(tau,
                        st["q25"].to_numpy(np.float64),
                        st["q75"].to_numpy(np.float64),
                        alpha=0.25, linewidth=0.0)
        ax.grid(True, alpha=0.25)

        if key == "intensity":
            ax.set_ylabel("Intensity (proxy)", fontsize=13)
            ax.set_title(title, fontsize=14, weight="bold")
        elif key == "tbmin":
            ax.set_ylabel("Tbmin (K)", fontsize=13)
        elif key == "area":
            ax.set_ylabel("Area (km$^2$)", fontsize=13)

        ax2 = ax.twinx()
        ax2.plot(tau, st["n"].to_numpy(np.float64), alpha=0.20, linewidth=1.2)
        ax2.set_ylabel("N", fontsize=10)
        ax2.tick_params(labelsize=9)

    axes[-1].set_xlabel("Normalized lifetime fraction (τ)", fontsize=13)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

def plot_peak_timing_hist(df_tau, var="intensity", track_col="track", bins=18, out_png="peak_timing_hist.png"):
    if var not in df_tau.columns:
        return

    rows = []
    for tid, g in df_tau.groupby(track_col, sort=True):
        g = g[np.isfinite(g["tau"]) & np.isfinite(g[var])]
        if len(g) < 2:
            continue
        if var == "tbmin":
            i = int(np.nanargmin(g[var].to_numpy(np.float64)))
        else:
            i = int(np.nanargmax(g[var].to_numpy(np.float64)))
        rows.append(float(g["tau"].iloc[i]))

    if len(rows) < 3:
        return

    x = np.asarray(rows, dtype=np.float64)
    fig, ax = plt.subplots(figsize=(8.0, 5.8))
    ax.hist(x, bins=bins, edgecolor="k", linewidth=0.7, alpha=0.9)
    ax.set_xlabel(f"τ at peak {var}" + (" (min)" if var == "tbmin" else " (max)"), fontsize=13)
    ax.set_ylabel("Number of MCS tracks", fontsize=13)
    ax.set_title(f"Peak Timing Across Lifecycle (τ) — {var}", fontsize=14, weight="bold")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

def plot_intense_probability_vs_tau(df_tau, intensity_col="intensity",
                                   tau_bin_col="tau_bin",
                                   n_bins=20, q=0.90,
                                   out_png="prob_intense_vs_tau.png"):
    if intensity_col not in df_tau.columns:
        return

    vals = df_tau[intensity_col].to_numpy(np.float64)
    vals = vals[np.isfinite(vals)]
    if vals.size < 10:
        return
    thr = float(np.nanquantile(vals, q))

    probs = []
    ns = []
    for b in range(n_bins):
        sub = df_tau[df_tau[tau_bin_col] == b]
        x = sub[intensity_col].to_numpy(np.float64)
        x = x[np.isfinite(x)]
        n = int(x.size)
        ns.append(n)
        if n == 0:
            probs.append(np.nan)
        else:
            probs.append(float(np.mean(x >= thr)))

    tau = (np.arange(n_bins) + 0.5) / n_bins
    probs = np.asarray(probs, dtype=np.float64)
    ns = np.asarray(ns, dtype=np.float64)

    fig, ax = plt.subplots(figsize=(8.2, 5.8))
    ax.plot(tau, probs, linewidth=2.2)
    ax.set_xlabel("Normalized lifetime fraction (τ)", fontsize=13)
    ax.set_ylabel(f"P(Intensity ≥ {q:.2f} quantile)", fontsize=13)
    ax.set_title("Probability of 'Intense' Proxy State vs Lifecycle (no precip)", fontsize=14, weight="bold")
    ax.grid(True, alpha=0.25)

    ax2 = ax.twinx()
    ax2.plot(tau, ns, alpha=0.20, linewidth=1.2)
    ax2.set_ylabel("N", fontsize=10)
    ax2.tick_params(labelsize=9)

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

# ---------------- MAIN ----------------
def main():
    if os.path.exists(DET_TMP):
        os.remove(DET_TMP)

    files = sorted(glob.glob(FILE_GLOB))
    meta = []
    for f in files:
        ts, h = parse_hour(f)
        if ts is not None:
            meta.append((ts, h, f))
    meta.sort()

    if not meta:
        print("[ERROR] No valid files found that match pattern/time.")
        return

    first_ds = xr.open_dataset(meta[0][2])
    try:
        Tb0, lat0, lon0, _ = extract_lat_lon_tb(first_ds, tb_var=TB_VAR, p_var=None)
    finally:
        first_ds.close()

    A0 = compute_area(lat0, lon0)
    last_shape = (lat0.size, lon0.size)

    ccs_count     = np.zeros(last_shape, dtype=np.int32)
    mcs_px_count  = np.zeros(last_shape, dtype=np.int32)
    mcs_int_sum   = np.zeros(last_shape, dtype=np.float32)

    prev = None
    nextID = 1
    file_lookup = {}

    # ---------------- FIRST PASS: detections + tracking ----------------
    for ts, h, f in meta:
        file_lookup[os.path.basename(f)] = f

        ds = xr.open_dataset(f)
        try:
            Tb, lat, lon, _ = extract_lat_lon_tb(ds, tb_var=TB_VAR, p_var=None)
        finally:
            ds.close()

        if (lat.size, lon.size) != last_shape or (not np.allclose(lat, lat0)) or (not np.allclose(lon, lon0)):
            raise RuntimeError(
                f"[ERROR] Grid changed between files. "
                f"First grid: {lat0.size}x{lon0.size}, this file: {lat.size}x{lon.size}. "
                f"Stop to avoid wrong density counts."
            )

        ccs_mask = np.isfinite(Tb) & (Tb < TB_CCS_THRESH)
        ccs_count += ccs_mask.astype(np.int32)

        lbl, num = label_objects(Tb)
        if A0.shape != Tb.shape or lbl.shape != Tb.shape:
            raise RuntimeError(f"Grid shape mismatch: A0{A0.shape}, Tb{Tb.shape}, lbl{lbl.shape}")

        if num == 0:
            prev = None
            del Tb, lbl
            gc.collect()
            continue

        records = []
        for oid in range(1, num+1):
            region = (lbl == oid)
            if not np.any(region):
                continue

            area_km2 = float(np.sum(A0[region]))
            if area_km2 < 200.0:
                continue

            tb_vals = Tb[region]
            if np.all(np.isnan(tb_vals)):
                continue
            tbmin = float(np.nanmin(tb_vals))

            iy, ix = np.where(region)
            cy = int(np.round(np.mean(iy))); cx = int(np.round(np.mean(ix)))
            cy = 0 if cy < 0 else (len(lat)-1 if cy >= len(lat) else cy)
            cx = 0 if cx < 0 else (len(lon)-1 if cx >= len(lon) else cx)
            clat, clon = float(lat[cy]), float(lon[cx])

            records.append((oid, area_km2, tbmin, clat, clon))

        del Tb, lbl
        gc.collect()

        if not records:
            prev = None
            continue

        cur = pd.DataFrame.from_records(records, columns=["oid","area","tbmin","clat","clon"])
        cur["area"]  = cur["area"].astype("float32")
        cur["tbmin"] = cur["tbmin"].astype("float32")
        cur["clat"]  = cur["clat"].astype("float32")
        cur["clon"]  = cur["clon"].astype("float32")

        cur["time"]  = pd.to_datetime(ts).floor("h")
        cur["hstr"]  = h
        cur["file"]  = os.path.basename(f)
        cur["track"] = -1

        if prev is not None and len(prev) > 0:
            used = set()
            cur_lat = cur["clat"].to_numpy()
            cur_lon = cur["clon"].to_numpy()
            for _, prow in prev.iterrows():
                d = haversine(prow.clat, prow.clon, cur_lat, cur_lon)
                j = int(np.argmin(d))
                if (j not in used) and (d[j] <= MAX_LINK_DIST_KM):
                    cur.iat[j, cur.columns.get_loc("track")] = int(prow.track)
                    used.add(j)

        unlinked = cur.index[cur["track"] == -1]
        if len(unlinked) > 0:
            cur.loc[unlinked, "track"] = np.arange(nextID, nextID + len(unlinked), dtype=np.int64)
            nextID += len(unlinked)

        write_chunk(cur[["oid","area","tbmin","clat","clon","time","hstr","file","track"]], DET_TMP)

        prev = cur[["clat","clon","track"]].copy()
        prev["clat"]  = prev["clat"].astype("float32")
        prev["clon"]  = prev["clon"].astype("float32")
        prev["track"] = prev["track"].astype("int64")

        del cur
        gc.collect()

    if not os.path.exists(DET_TMP):
        print("[WARN] No detections were found.")
        return

    dtypes = {
        "oid":"int32","area":"float32","tbmin":"float32","clat":"float32","clon":"float32",
        "hstr":"string","file":"string","track":"int64"
    }
    det = pd.read_csv(DET_TMP, dtype=dtypes, parse_dates=["time"])
    det["time"] = pd.to_datetime(det["time"]).dt.floor("h")
    det["time_hour"] = det["time"]

    trk = det.groupby("track", sort=True).agg(
        start_time=("time","min"),
        end_time=("time","max"),
        dur=("time_hour","nunique"),
        maxA=("area","max"),
        minTb=("tbmin","min"),
    )
    trk["mcs"] = (trk["maxA"] >= MCS_MIN_AREA_KM2) & (trk["dur"] >= MCS_MIN_DURATION_H) & (trk["minTb"] <= TB_CORE_THRESH)

    keep_ids = trk.index[trk["mcs"]].to_numpy()
    det = det[det["track"].isin(keep_ids)].copy()
    trk = trk.loc[keep_ids].copy()

    if det.empty:
        print("[INFO] No tracks met MCS criteria with current thresholds.")
        return

    # renumber tracks
    renum = {old: new for new, old in enumerate(sorted(keep_ids), start=1)}
    det["track"] = det["track"].map(renum).astype("int32")
    trk.index = trk.index.map(renum)

    # intensity proxy (per-object, per-hour)
    det["intensity"] = ((TB_CCS_THRESH - det["tbmin"]) / 50.0).clip(lower=0) + (det["area"] / (det["area"] + 40000.0))
    det["intensity"] = det["intensity"].astype("float32")

    det.to_csv(DET_OUT, index=False)
    trk.to_csv(TRK_OUT)

    det = det.sort_values(["track", "time"]).copy()
    det_centroids = det.drop_duplicates(subset=["track", "time_hour"], keep="first").copy()

    # ---------------- Build lookups for SECOND PASS ----------------
    oids_by_filehour = {}
    for (fname, th), sub in det.groupby(["file", "time_hour"]):
        oids_by_filehour[(fname, th)] = set(sub["oid"].astype(int).tolist())

    inten_by_filehour_oid = {}
    for (fname, th), sub in det.groupby(["file", "time_hour"]):
        for _, rr in sub.iterrows():
            inten_by_filehour_oid[(fname, th, int(rr["oid"]))] = float(rr["intensity"])

    # =======================
    # OVERVIEW STATS
    # =======================
    os.makedirs(FIG_DIR_OV, exist_ok=True)

    life = compute_mcs_lifecycle_stats(det_centroids)
    life_csv = os.path.join(FIG_DIR_OV, "mcs_lifecycle_summary.csv")
    life.to_csv(life_csv, index=False)

    _histplot(life["duration_h"], "Duration (hours)",
             "MCS Lifecycle: Duration Distribution",
             os.path.join(FIG_DIR_OV, "hist_duration_h.png"))

    _histplot(life["mean_speed_kmh"], "Mean Translational Speed (km h$^{-1}$)",
             "MCS Lifecycle: Mean Speed Distribution",
             os.path.join(FIG_DIR_OV, "hist_mean_speed_kmh.png"))

    _histplot(life["max_area_km2"], "Maximum Area (km$^2$)",
             "MCS Lifecycle: Maximum Area Distribution",
             os.path.join(FIG_DIR_OV, "hist_max_area_km2.png"))

    _histplot(life["min_tb_k"], "Minimum Tb (K)",
             "MCS Lifecycle: Minimum Tb Distribution",
             os.path.join(FIG_DIR_OV, "hist_min_tb_k.png"))

    _scatter_with_regression(
        life["duration_h"], life["max_area_km2"],
        "Duration (hours)", "Max Area (km$^2$)",
        "Duration vs Max Area (per tracked MCS)",
        os.path.join(FIG_DIR_OV, "scatter_duration_vs_maxarea.png")
    )

    _scatter_with_regression(
        life["duration_h"], life["peak_intensity"],
        "Duration (hours)", "Peak Intensity",
        "Duration vs Peak Intensity (per tracked MCS)",
        os.path.join(FIG_DIR_OV, "scatter_duration_vs_peakintensity.png")
    )

    _scatter_with_regression(
        life["mean_speed_kmh"], life["peak_intensity"],
        "Mean Speed (km h$^{-1}$)", "Peak Intensity",
        "Mean Speed vs Peak Intensity (per tracked MCS)",
        os.path.join(FIG_DIR_OV, "scatter_meanspeed_vs_peakintensity.png")
    )

    _scatter_with_regression(
        det_centroids["area"], det_centroids["intensity"],
        "Area (km$^2$) at time", "Intensity at time",
        "Hourly Samples: Area vs Intensity",
        os.path.join(FIG_DIR_OV, "scatter_hourly_area_vs_intensity.png")
    )

    _scatter_with_regression(
        det_centroids["tbmin"], det_centroids["intensity"],
        "Tbmin (K) at time", "Intensity at time",
        "Hourly Samples: Tbmin vs Intensity",
        os.path.join(FIG_DIR_OV, "scatter_hourly_tbmin_vs_intensity.png")
    )

    print(f"[OK] Lifecycle stats CSV: {life_csv}")
    print(f"[OK] Overview stats figures saved in: {FIG_DIR_OV}/")

    # =======================
    # Lifecycle intensification diagnostics (NO precip)
    # =======================
    tau, comp, df_tau = lifecycle_composite(det_centroids, n_bins=LIFECYCLE_NBINS)

    out_life_comp = os.path.join(FIG_DIR_OV, "lifecycle_composite_intensity_tb_area.png")
    plot_lifecycle_composites(
        tau, comp, out_life_comp,
        title="MCS Lifecycle Composite (Intensity/Tbmin/Area; no precipitation used)",
        show_intensity=True, show_tb=True, show_area=True
    )

    out_peak_inten = os.path.join(FIG_DIR_OV, "peak_timing_tau_hist_intensity.png")
    plot_peak_timing_hist(df_tau, var="intensity", bins=18, out_png=out_peak_inten)

    out_peak_tb = os.path.join(FIG_DIR_OV, "peak_timing_tau_hist_tbmin.png")
    plot_peak_timing_hist(df_tau, var="tbmin", bins=18, out_png=out_peak_tb)

    out_prob_intense = os.path.join(FIG_DIR_OV, "prob_intense_vs_tau.png")
    plot_intense_probability_vs_tau(
        df_tau,
        intensity_col="intensity",
        tau_bin_col="tau_bin",
        n_bins=LIFECYCLE_NBINS,
        q=INTENSE_Q,
        out_png=out_prob_intense
    )

    print(f"[OK] Lifecycle composite (no precip): {out_life_comp}")
    print(f"[OK] Peak timing hist (intensity):    {out_peak_inten}")
    print(f"[OK] Peak timing hist (tbmin):        {out_peak_tb}")
    print(f"[OK] P(intense|tau) (no precip):      {out_prob_intense}")

    # =======================
    # SECOND PASS: accumulate pixels + accumulate intensity in true MCS objects
    # =======================
    for ts, h, f in meta:
        base = os.path.basename(f)
        th = pd.to_datetime(ts).floor("h")
        key = (base, th)
        if key not in oids_by_filehour:
            continue

        ds = xr.open_dataset(f)
        try:
            Tb, lat, lon, _ = extract_lat_lon_tb(ds, tb_var=TB_VAR, p_var=None)
        finally:
            ds.close()

        lbl, num = label_objects(Tb)
        keep_oids = oids_by_filehour[key]

        for oid in keep_oids:
            if oid <= 0 or oid > num:
                continue

            region = (lbl == oid)
            if not np.any(region):
                continue

            mcs_px_count += region.astype(np.int32)

            inten = inten_by_filehour_oid.get((base, th, int(oid)), np.nan)
            if np.isfinite(inten):
                mcs_int_sum[region] += np.float32(inten)

        del Tb, lbl
        gc.collect()

    # KDE fields
    lon_grid = lon0.copy()
    lat_grid = lat0.copy()

    track_Z = _kde_on_grid(det_centroids["clon"].values,
                           det_centroids["clat"].values,
                           lon_grid, lat_grid,
                           bw_adjust=KDE_BW_ADJUST)

    heavy_Z = None
    if HAS_PRECIP:
        rng = np.random.default_rng(42)
        heavy_lon_all = []
        heavy_lat_all = []
        for ts, h, f in meta:
            ds = xr.open_dataset(f)
            try:
                _, lat, lon, P = extract_lat_lon_tb(ds, tb_var=TB_VAR, p_var=PRECIP_VAR)
            finally:
                ds.close()

            if P is None:
                continue

            mask = np.isfinite(P) & (P >= PRECIP_KDE_THRESH)
            if not np.any(mask):
                del P, mask
                gc.collect()
                continue

            iy, ix = np.where(mask)
            n = iy.size
            if n > MAX_PTS_PER_HOUR:
                sel = rng.choice(n, size=MAX_PTS_PER_HOUR, replace=False)
                iy = iy[sel]; ix = ix[sel]

            heavy_lat_all.append(lat[iy].astype(np.float64))
            heavy_lon_all.append(lon[ix].astype(np.float64))

            del P, mask
            gc.collect()

        if len(heavy_lon_all) > 0:
            heavy_lon_all = np.concatenate(heavy_lon_all)
            heavy_lat_all = np.concatenate(heavy_lat_all)
            heavy_Z = _kde_on_grid(heavy_lon_all, heavy_lat_all, lon_grid, lat_grid, bw_adjust=KDE_BW_ADJUST)

    # OVERVIEW map (centroid intensity + KDE contours)
    os.makedirs(FIG_DIR_OV, exist_ok=True)
    fig, ax, pc = make_map("MCS Intensity with Track KDE and Heavy-Precip KDE")
    add_latlon_ticks(ax, fontsize=16)

    I = det_centroids["intensity"].to_numpy(np.float32)

    # ---- NEW: safe scaling (avoid divide-by-zero if all values identical) ----
    Imin = float(np.nanmin(I))
    Imax = float(np.nanmax(I))
    den = (Imax - Imin)
    if (not np.isfinite(den)) or den <= 0:
        I_scaled = np.full_like(I, 120.0, dtype=np.float32)
    else:
        I_scaled = 40.0 + 260.0 * ((I - Imin) / den)

    sc = ax.scatter(det_centroids["clon"], det_centroids["clat"], transform=pc,
                    s=I_scaled, c=I, cmap="Blues",
                    edgecolor="black", linewidth=0.5, alpha=0.85, zorder=30)

    lv_track = _safe_levels(track_Z, KDE_LEVELS)
    if (track_Z is not None) and (lv_track is not None):
        ax.contour(lon_grid, lat_grid, track_Z,
                   levels=lv_track, colors="black", linewidths=2.4,
                   transform=pc, zorder=40)

    lv_heavy = _safe_levels(heavy_Z, KDE_LEVELS)
    if (heavy_Z is not None) and (lv_heavy is not None):
        ax.contour(lon_grid, lat_grid, heavy_Z,
                   levels=lv_heavy, colors="red", linewidths=2.2, linestyles="--",
                   transform=pc, zorder=41)

    for city, (clon, clat) in CITIES.items():
        ax.plot(clon, clat, 'kD', transform=pc, zorder=60)
        ax.text(clon - 0.5, clat - 0.5, city, transform=pc,
                fontsize=14, color='k', zorder=61)

    cb_ax = fig.add_axes([0.80, 0.35, 0.02, 0.50])
    cb = plt.colorbar(sc, cax=cb_ax)
    cb.set_label("MCS Intensity Index", fontsize=16)
    cb.ax.tick_params(labelsize=16)

    legend_handles = [
        Line2D([0], [0], color="black", lw=2.4, label="MCS Track KDE (normalized 0–1)"),
        Line2D([0], [0], color="red",   lw=2.2, ls="--",
               label=rf"Hourly Precip KDE (P ≥ {PRECIP_KDE_THRESH:g} mm h$^{{-1}}$, normalized 0–1)")
    ]
    ax.legend(handles=legend_handles,
              loc="lower center", bbox_to_anchor=(0.5, -0.17),
              ncol=2, fontsize=14, frameon=True, framealpha=0.95)

    fig.subplots_adjust(left=0.06, right=0.865, top=0.93, bottom=0.25)
    out_overview = os.path.join(FIG_DIR_OV, "mcs_intensity_with_kde_contours_labeled.png")
    plt.savefig(out_overview, dpi=300, bbox_inches="tight")
    plt.close(fig)

    # =======================
    # DENSITY MAPS
    # =======================
    out_mcs_px = os.path.join(FIG_DIR_OV, "density_ALLpixels_in_TRUE_MCS_objects_counts.png")
    save_density_map(
        mcs_px_count, lon0, lat0,
        "Density: All Grid Points Inside TRUE MCS Tracked Objects",
        out_mcs_px,
        cmap="gist_ncar",
        vmin=0.0, vmax=100.0,
        cbar_label="Density Grid Map"
    )

    # ---- FIX: define out_ccs (previously printed but never created) ----
    out_ccs = os.path.join(FIG_DIR_OV, "density_CCS_pixels_TbLT241K_counts.png")
    save_density_map(
        ccs_count, lon0, lat0,
        f"Density: Cloud Shield Pixels (Tb < {TB_CCS_THRESH:g} K)",
        out_ccs,
        cmap="gist_ncar",
        vmin=0.0, vmax=100.0,
        cbar_label="Density Grid Map (# hours)"
    )

    # grid-mean intensity map (derived from the same TRUE-object pixels)
    mcs_int_mean = np.full_like(mcs_int_sum, np.nan, dtype=np.float32)
    ok = mcs_px_count > 0
    mcs_int_mean[ok] = mcs_int_sum[ok] / mcs_px_count[ok].astype(np.float32)

    out_mcs_int_grid = os.path.join(FIG_DIR_OV, "intensity_grid_TRUE_MCS_objects_mean.png")
    save_grid_intensity_map(
        mcs_int_mean, lon0, lat0,
        "Grid Mean Intensity: TRUE MCS Object Pixels (time-averaged proxy)",
        out_mcs_int_grid,
        cmap="gist_ncar",
        vmin=None, vmax=None,
        cbar_label="Grid Mean Intensity (proxy)"
    )

    # ==========================================================
    # MULTIPLIED MAP = density * mean_intensity (== mcs_int_sum)
    # ==========================================================
    mcs_mult = mcs_px_count.astype(np.float32) * mcs_int_mean  # == cumulative intensity sum

    out_mcs_mult = os.path.join(FIG_DIR_OV, "mult_density_times_meanIntensity_TRUE_MCS_objects.png")
    save_grid_intensity_map(
        mcs_mult, lon0, lat0,
        "TRUE MCS Object Pixels: Density × Grid-Mean Intensity (== Cumulative Intensity)",
        out_mcs_mult,
        cmap="gist_ncar",
        vmin=None, vmax=None,
        cbar_label="Cumulative Intensity (proxy)"
    )

    # ==========================================================
    # COMPANION 1: Normalized cumulative intensity (0–1)
    # + Extreme precip KDE contours
    # ==========================================================
    mcs_mult_norm = np.full_like(mcs_mult, np.nan, dtype=np.float32)
    finite = np.isfinite(mcs_mult)
    if np.any(finite):
        mmin = float(np.nanmin(mcs_mult[finite]))
        mmax = float(np.nanmax(mcs_mult[finite]))
        mcs_mult_norm[finite] = (mcs_mult[finite] - mmin) / (mmax - mmin + 1e-6)

    out_mcs_mult_norm = os.path.join(
        FIG_DIR_OV,
        "mult_density_times_meanIntensity_TRUE_MCS_objects_NORMALIZED.png"
    )
    save_grid_intensity_map(
        mcs_mult_norm, lon0, lat0, "Convoluted MCS track density and mean intensity",
        out_mcs_mult_norm,
        cmap="gist_ncar",
        vmin=0.0, vmax=1.0,
        cbar_label="Normalized MCS Cumulative Intensity (0–1; density × mean intensity)",
        contour_Z=heavy_Z,
        contour_levels=KDE_LEVELS,
        contour_color="black",
        contour_ls="--",
        contour_lw=2.4,
        contour_label=rf"Extreme precip KDE (P ≥ {PRECIP_KDE_THRESH:g} mm h$^{{-1}}$, normalized 0–1)"
    )
    
    out_scatter_density_vs_meanI = os.path.join(
        FIG_DIR_OV,
        "scatter_density_vs_meanIntensity_TRUE_MCS_objects.png"
    )
    
    scatter_grid_with_regression(
        mcs_px_count,
        mcs_int_mean,
        xlabel="MCS Track Density (# hours)",
        ylabel="Mean MCS Intensity (proxy)",
        title="MCS Track Density vs Mean Intensity",
        out_png=out_scatter_density_vs_meanI
    )
    
    if heavy_Z is not None:
        out_scatter_mult_vs_precip = os.path.join(
            FIG_DIR_OV,
            "scatter_normCumulativeIntensity_vs_extremePrecipKDE.png"
        )
    
        scatter_grid_with_regression(
            mcs_mult_norm,
            heavy_Z,
            xlabel="Normalized MCS Cumulative Intensity (0–1)",
            ylabel=f"Extreme Precip KDE (P ≥ {PRECIP_KDE_THRESH:g} mm h$^{{-1}}$, 0–1)",
            title="MCS Cumulative Intensity vs Extreme Precipitation Hotspots",
            out_png=out_scatter_mult_vs_precip
        )

    if heavy_Z is not None:

        # -----------------------------------------
        # 1) Density vs Extreme Precipitation KDE
        # -----------------------------------------
        out_scatter_density_vs_precip = os.path.join(
            FIG_DIR_OV,
            "scatter_density_vs_extremePrecipKDE.png"
        )

        scatter_grid_with_regression(
            mcs_px_count,
            heavy_Z,
            xlabel="MCS Track Density (# hours)",
            ylabel=f"Extreme Precip KDE (P ≥ {PRECIP_KDE_THRESH:g} mm h$^{{-1}}$, 0–1)",
            title="MCS Track Density vs Extreme Precipitation Hotspots",
            out_png=out_scatter_density_vs_precip,
            stats_loc="topleft"
        )


        # -----------------------------------------
        # 2) Mean Intensity vs Extreme Precipitation KDE
        # -----------------------------------------
        out_scatter_meanI_vs_precip = os.path.join(
            FIG_DIR_OV,
            "scatter_meanIntensity_vs_extremePrecipKDE.png"
        )

        scatter_grid_with_regression(
            mcs_int_mean,
            heavy_Z,
            xlabel="Mean MCS Intensity (proxy)",
            ylabel=f"Extreme Precip KDE (P ≥ {PRECIP_KDE_THRESH:g} mm h$^{{-1}}$, 0–1)",
            title="Mean MCS Intensity vs Extreme Precipitation Hotspots",
            out_png=out_scatter_meanI_vs_precip,
            stats_loc="topleft"
        )

    # =======================
    # SNAPSHOTS
    # =======================
    os.makedirs(FIG_DIR_SNP, exist_ok=True)

    precip_cmap = ListedColormap(PRECIP_COLOURS)
    precip_norm = BoundaryNorm(PRECIP_LEVELS, precip_cmap.N, clip=True)
    contour_lvls = [l for l in PRECIP_LEVELS if l % 2 == 0 and l > 0]

    start_times = det.groupby("track")["time"].min()
    track_label = {tid: f"MCS {tid:02d} ({start_times.loc[tid].strftime('%Y%m%d-%H')})" for tid in start_times.index}

    for tid, track_df in det.groupby("track", sort=True):
        track_df = track_df.sort_values("time").copy()
        track_df["step"] = np.arange(1, len(track_df) + 1)
        mcs_name = track_label[tid]

        start_row  = track_df.iloc[0]
        start_lon  = float(start_row["clon"])
        start_lat  = float(start_row["clat"])
        start_text = f"MCS {int(tid):02d} Start"

        track_pts = track_df[["clon","clat"]].to_numpy(np.float64)
        city_pts  = np.array([[v[0], v[1]] for v in CITIES.values()], dtype=np.float64) if len(CITIES) else np.empty((0,2))
        avoid_pts = np.vstack([track_pts, city_pts]) if (track_pts.size or city_pts.size) else np.empty((0,2))

        used_label_anchors = []

        for _, r in track_df.iterrows():
            nc_path = file_lookup.get(r["file"])
            if nc_path is None:
                continue

            ds = xr.open_dataset(nc_path)
            try:
                Tb, lat, lon, P = extract_lat_lon_tb(ds, tb_var=TB_VAR, p_var=PRECIP_VAR if HAS_PRECIP else None)
            finally:
                ds.close()

            mask_ccs = (Tb < TB_CCS_THRESH)
            Tb_ccs = np.where(mask_ccs, Tb, np.nan)

            time_label = pd.to_datetime(r["time"]).strftime("%Y-%m-%d %H UTC")
            title = f"{mcs_name} | {time_label} | Step {int(r['step']):02d}"
            fig, ax, pc = make_map(title)
            add_latlon_ticks(ax, fontsize=16)

            fig.subplots_adjust(left=0.06, right=0.94, top=0.93, bottom=0.32)

            # precip
            m_prec = None
            if P is not None:
                m_prec = ax.pcolormesh(lon, lat, P, transform=pc,
                                       cmap=precip_cmap, norm=precip_norm,
                                       shading="auto", alpha=1.0, zorder=3)

            # CCS Tb overlay
            m_obj = ax.pcolormesh(lon, lat, Tb_ccs, transform=pc, cmap="Blues",
                                  vmin=190, vmax=240, shading="auto", alpha=0.35, zorder=5)

            # object boundary ONLY (no fill)
            if SHOW_OBJECT_PIXELS_IN_SNAPSHOTS:
                lbl, num = label_objects(Tb)
                oid_now = int(r["oid"])
            
                if 1 <= oid_now <= num:
                    obj_mask = (lbl == oid_now)
            
                    # Draw ONLY the boundary in black
                    ax.contour(
                        lon, lat,
                        obj_mask.astype(np.int8),
                        levels=[0.5],
                        colors='blue',       # <-- blue border
                        linewidths=1.5,       # adjust thickness if needed
                        transform=pc,
                        zorder=4
                    )
            
                del lbl
                gc.collect()

            # track path/markers
            hist = track_df[track_df["time"] <= r["time"]]
            ax.plot(hist["clon"], hist["clat"], transform=pc, color="black",
                    linestyle=":", linewidth=2.0, zorder=7)
            ax.scatter(hist["clon"], hist["clat"], transform=pc, s=20, color="black",
                       edgecolor="black", linewidth=0.5, zorder=8)

            for _, pt in hist.iterrows():
                ax.text(pt["clon"] + 0.08, pt["clat"] + 0.08, f"{int(pt['step']):02d}",
                        fontsize=9, weight="bold", color="black", transform=pc, zorder=9)

            ax.scatter(r["clon"], r["clat"], transform=pc, s=40, color="yellow",
                       edgecolor="black", linewidth=1.0, zorder=10)

            # start marker + label
            if PLOT_START_ON_ALL_STEPS or int(r["step"]) == 1:
                ax.scatter(start_lon, start_lat, transform=pc, s=160,
                           marker="*", color="lime",
                           edgecolor="black", linewidth=1.0, zorder=9)

                tx, ty = choose_nonoverlap_label_anchor(
                    start_lon, start_lat,
                    track_lonlat=avoid_pts,
                    used_label_anchors=used_label_anchors,
                    extent=MAP_EXTENT
                )
                used_label_anchors.append([tx, ty])

                ax.annotate(
                    start_text,
                    xy=(start_lon, start_lat), xycoords=pc._as_mpl_transform(ax),
                    xytext=(tx, ty), textcoords=pc._as_mpl_transform(ax),
                    fontsize=START_TEXT_FS, weight="bold", color=START_TEXT_COLOR,
                    ha="left", va="bottom",
                    arrowprops=dict(arrowstyle="->", lw=START_ARROW_LW, color=START_TEXT_COLOR),
                    zorder=13
                )

            # cities
            for city, (clon, clat) in CITIES.items():
                ax.plot(clon, clat, 'kD', transform=pc, zorder=11)
                ax.text(clon - 0.5, clat - 0.5, city, transform=pc,
                        fontsize=12, color='k', zorder=12)

            # COLORBARS
            cax_obj = fig.add_axes([0.83, 0.36, 0.02, 0.50])
            cb_obj = plt.colorbar(m_obj, cax=cax_obj)
            cb_obj.set_label("MCS Tb (K)", fontsize=16)
            cb_obj.ax.tick_params(labelsize=16, length=0)
            cb_obj.outline.set_linewidth(0)

            if m_prec is not None:
                cax_p = fig.add_axes([0.12, 0.25, 0.76, 0.028])
                cb_p = plt.colorbar(m_prec, cax=cax_p, orientation='horizontal',
                                    ticks=PRECIP_LEVELS[::2], extend='max')
                cb_p.set_label(f"Precipitation ({PRECIP_UNITS})", fontsize=16)
                cb_p.ax.tick_params(labelsize=16)

            obj_handle   = Line2D([0], [0], color=OBJ_EDGE_COLOR, lw=OBJ_EDGE_LW,
                                  label="MCS Object Grids (this hour)")
            start_handle = Line2D([0], [0], marker='*', color='w',
                                  markerfacecolor='lime', markeredgecolor='k',
                                  markersize=14, linestyle='None', label="MCS Start")
            path_handle  = Line2D([0], [0], color="black", lw=2.0, ls=":", label="MCS Path")
            cur_handle   = Line2D([0], [0], marker='o', color='w',
                                  markerfacecolor='yellow', markeredgecolor='k',
                                  markersize=10, linestyle='None', label="Current Centroid")

            fig.legend(
                handles=[path_handle, obj_handle, cur_handle, start_handle],
                loc="lower center",
                bbox_to_anchor=(0.5, 0.080),
                ncol=2,
                fontsize=15,
                frameon=True,
                framealpha=0.95,
                columnspacing=1.8,
                handlelength=2.2,
                borderaxespad=0.2
            )

            safe_name = mcs_name.replace(" ", "_").replace("(", "").replace(")", "")
            out_png = os.path.join(FIG_DIR_SNP, f"{safe_name}_step_{int(r['step']):02d}.png")
            plt.savefig(out_png, dpi=300, bbox_inches="tight")
            plt.close(fig)

            del Tb, Tb_ccs, P
            gc.collect()

    print(f"[OK] Overview saved: {out_overview}")
    print(f"[OK] Pixel density (TRUE MCS objects): {out_mcs_px}")
    print(f"[OK] Grid mean intensity (TRUE MCS objects): {out_mcs_int_grid}")
    print(f"[OK] Multiplied map (cumulative intensity): {out_mcs_mult}")
    print(f"[OK] Normalized cumulative map (with extreme precip KDE contours): {out_mcs_mult_norm}")
    print(f"[OK] Pixel density (Tb < {TB_CCS_THRESH:g}K): {out_ccs}")
    print(f"[OK] Detections CSV: {DET_OUT}")
    print(f"[OK] Tracks CSV: {TRK_OUT}")

if __name__ == "__main__":
    main()

ValueError: Cannot calculate a linear regression if all x values are identical

In [10]:
import warnings; warnings.filterwarnings("ignore")

import os, re, glob, gc
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

from scipy import ndimage as ndi
from scipy.stats import gaussian_kde, linregress, pearsonr, spearmanr

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D

# ---------------- USER SETTINGS ----------------
FILE_GLOB = "/glade/work/sshohan/MPAS_MCS_Tracking/hourly_subset/regridded_to_MERGIR/regridded*.nc"
TB_VAR = "ctt"

# Precip overlay for snapshots
HAS_PRECIP = True
PRECIP_VAR = "prec_total"
PRECIP_UNITS = "mm h$^{-1}$"

# KDE overlay for OVERVIEW
PRECIP_KDE_THRESH = 20.0
MAX_PTS_PER_HOUR  = 12000
KDE_LEVELS        = [0.2, 0.4, 0.6, 0.8]  # normalized 0–1 contour levels
KDE_BW_ADJUST     = 1.00
# --- Snapshot start-marker settings ---
PLOT_START_ON_ALL_STEPS = True   # if False, only marks on step 1

# Smart placement controls (reduce overlaps with track points/text already placed)
LABEL_MIN_DIST_DEG   = 0.50
LABEL_MIN_DIST_TEXT  = 0.65
LABEL_MIN_EDGE_PAD   = 0.35
LABEL_TRY_RADII_DEG  = [0.55, 0.85, 1.15, 1.45, 1.80]

# label style
START_TEXT_COLOR = "deeppink"
START_TEXT_FS    = 14
START_ARROW_LW   = 1.8

# --- Snapshot: show full MCS object pixels (distinguishable from centroid) ---
SHOW_OBJECT_PIXELS_IN_SNAPSHOTS = True
OBJ_PIX_COLOR  = "magenta"   # object pixels fill
OBJ_PIX_ALPHA  = 0.22        # transparency
OBJ_EDGE_COLOR = "magenta"   # outline color
OBJ_EDGE_LW    = 2.0

# Provided precip colors/levels
PRECIP_COLOURS = ['white','#00EEEE','#00B2EE','#1E90FF','#1f77b4','#104E8B','#7FFF00',
                  '#00CD00',"#53a447",'#008B00','#FFFF00','#FFD700','#CD8500','#FF7F00',
                  '#EE4000','#CD0000','#8B0000','#8968CD','#912CEE','#8B008B']
PRECIP_LEVELS = [4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42]

TB_CCS_THRESH = 241.0
TB_CORE_THRESH = 225.0
MCS_MIN_AREA_KM2 = 60000.0
MCS_MIN_DURATION_H = 6

CONNECTIVITY = 1
MAX_LINK_DIST_KM = 250.0

DET_OUT = "mcs_detections.csv"
TRK_OUT = "mcs_tracks.csv"
FIG_DIR_OV = "figs_overview"
FIG_DIR_SNP = "figs_snapshots"
DET_TMP = "_tmp_detections_stream.csv"
# Snapshot map extent (must match make_map)
MAP_EXTENT = (-107.42619, -91.85856, 25.02549, 37.27808)  # (lonmin, lonmax, latmin, latmax)

CITIES = {
    'Dallas':  (-96.48, 32.46),
    'Houston': (-95.22, 29.45),
    'Austin':  (-97.44, 30.16),
}

# --- NEW: lifecycle diagnostics (no precip) ---
LIFECYCLE_NBINS = 20
INTENSE_Q = 0.90  # "severe proxy" threshold = top 10% intensity across all hourly samples

def parse_hour(fname):
    """
    Parse time from NetCDF file (NOT filename).

    Returns
    -------
    ts : pandas.Timestamp or None
        Timestamp for the file hour
    key : str or None
        Formatted key, e.g. '20150508-16'
    """
    try:
        with xr.open_dataset(fname) as ds:
            if "time" not in ds.variables or ds["time"].size == 0:
                return None, None

            # Read time and convert to pandas datetime
            ts = pd.to_datetime(ds["time"].values[0])

            if pd.isna(ts):
                return None, None

            # Round/floor to hour (important for consistency)
            ts = pd.Timestamp(ts).floor("h")

            return ts, ts.strftime("%Y%m%d-%H")

    except Exception as e:
        print(f"[WARN] Failed to read time from {fname}: {e}")
        return None, None

def extract_lat_lon_tb(ds, tb_var=TB_VAR, p_var=None):
    da = ds[tb_var].isel(time=0)
    dims = list(da.dims)

    # spatial dims from Tb field
    if len(dims) < 2:
        raise ValueError(f"{tb_var} does not have at least 2 spatial dims: {dims}")
    lat_dim, lon_dim = dims[-2], dims[-1]

    # Prefer true geolocation variables if present
    if "lat" in ds.variables and "lon" in ds.variables:
        latv = ds["lat"].values
        lonv = ds["lon"].values
    elif "latitude" in ds.variables and "longitude" in ds.variables:
        latv = ds["latitude"].values
        lonv = ds["longitude"].values
    else:
        raise KeyError("Could not find 2D/1D latitude and longitude variables in dataset")

    # Convert 2D lat/lon to 1D axes if rectilinear after regridding
    if latv.ndim == 2 and lonv.ndim == 2:
        lat1d = latv[:, 0]
        lon1d = lonv[0, :]
    elif latv.ndim == 1 and lonv.ndim == 1:
        lat1d = latv
        lon1d = lonv
    else:
        raise ValueError("Unsupported lat/lon dimensionality")

    Tb2d = da.transpose(lat_dim, lon_dim).astype(np.float32).values

    P2d = None
    if (p_var is not None) and (p_var in ds.variables):
        P2d = ds[p_var].isel(time=0).transpose(lat_dim, lon_dim).astype(np.float32).values

    return Tb2d, lat1d.astype(np.float32), lon1d.astype(np.float32), P2d

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    d = np.deg2rad
    lat1 = d(lat1); lon1 = d(lon1)
    lat2 = d(lat2); lon2 = d(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return 2.0*R*np.arcsin(np.sqrt(a))

def label_objects(Tb2d):
    mask = np.isfinite(Tb2d) & (Tb2d < TB_CCS_THRESH) & (Tb2d > 150.0) & (Tb2d < 350.0)
    lbl, num = ndi.label(mask, ndi.generate_binary_structure(2, CONNECTIVITY))
    return lbl, num

def compute_area(lat, lon):
    lat = np.asarray(lat, dtype=np.float32)
    lon = np.asarray(lon, dtype=np.float32)
    if lat.size < 2 or lon.size < 2:
        raise RuntimeError("[ERROR] compute_area requires at least 2 lat and 2 lon points.")
    dlat = float(np.abs(lat[1]-lat[0]))
    dlon = float(np.abs(lon[1]-lon[0]))
    km_per_deg_lat = 111.32
    km_per_deg_lon = 111.32 * np.cos(np.deg2rad(lat))
    cell_area_row = (km_per_deg_lat * dlat) * (km_per_deg_lon * dlon)   # (nlat,)
    return (cell_area_row[:, None] * np.ones((1, lon.size), dtype=np.float32)).astype(np.float32)

def add_latlon_ticks(ax, fontsize=14):
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.0)
    gl.xlines = False
    gl.ylines = False
    gl.top_labels = False
    gl.right_labels = False
    try:
        gl.xlabel_style = {"size": fontsize}
        gl.ylabel_style = {"size": fontsize}
    except Exception:
        pass

def make_map(title):
    proj = ccrs.PlateCarree()
    fig, ax = plt.subplots(figsize=(11.2, 9.0), subplot_kw=dict(projection=proj))
    ax.set_extent(list(MAP_EXTENT), crs=proj)
    ax.set_title(title, fontsize=16, weight="bold")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='k', zorder=50)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='k', zorder=51)
    try:
        ax.add_feature(cfeature.STATES.with_scale("10m"), linewidth=0.7, edgecolor='k',
                       facecolor='none', zorder=52)
    except Exception:
        ax.add_feature(cfeature.STATES, linewidth=0.7, edgecolor='k', facecolor='none', zorder=52)
    return fig, ax, proj

def write_chunk(df, path):
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=header, index=False)

def _kde_on_grid(lon_pts, lat_pts, lon_grid_1d, lat_grid_1d, bw_adjust=1.0):
    lon_pts = np.asarray(lon_pts, dtype=np.float64)
    lat_pts = np.asarray(lat_pts, dtype=np.float64)
    ok = np.isfinite(lon_pts) & np.isfinite(lat_pts)
    lon_pts = lon_pts[ok]; lat_pts = lat_pts[ok]
    if lon_pts.size < 10:
        return None

    values = np.vstack([lon_pts, lat_pts])
    kde = gaussian_kde(values)
    if bw_adjust != 1.0:
        kde.set_bandwidth(bw_method=kde.factor * bw_adjust)

    LON, LAT = np.meshgrid(lon_grid_1d.astype(np.float64), lat_grid_1d.astype(np.float64))
    positions = np.vstack([LON.ravel(), LAT.ravel()])
    Z = kde(positions).reshape(LAT.shape)

    zmax = np.nanmax(Z)
    if not np.isfinite(zmax) or zmax <= 0:
        return None
    return (Z / zmax).astype(np.float32)

# ---- NEW: safe contour levels helper (prevents contour() errors when Z has degenerate range) ----
def _safe_levels(Z, levels):
    if Z is None:
        return None
    zmin = float(np.nanmin(Z))
    zmax = float(np.nanmax(Z))
    if (not np.isfinite(zmin)) or (not np.isfinite(zmax)) or (zmax <= zmin):
        return None
    lv = [float(l) for l in levels if (float(l) > zmin) and (float(l) < zmax)]
    return lv if len(lv) else None

def choose_nonoverlap_label_anchor(start_lon, start_lat, track_lonlat, used_label_anchors,
                                  extent=MAP_EXTENT):
    lonmin, lonmax, latmin, latmax = extent
    directions = [
        ( 1,  1), (-1,  1), ( 1, -1), (-1, -1),
        ( 1,  0), (-1,  0), ( 0,  1), ( 0, -1),
    ]
    candidates = []
    for r in LABEL_TRY_RADII_DEG:
        for sx, sy in directions:
            candidates.append((sx*r, sy*r))
    candidates += [(2.3, 1.1), (-2.3, 1.1), (2.3, -1.1), (-2.3, -1.1)]

    track_lonlat = np.asarray(track_lonlat, dtype=np.float64)
    used_label_anchors = np.asarray(used_label_anchors, dtype=np.float64) if len(used_label_anchors) else None

    for dx, dy in candidates:
        tx = start_lon + dx
        ty = start_lat + dy
        if (tx < lonmin + LABEL_MIN_EDGE_PAD) or (tx > lonmax - LABEL_MIN_EDGE_PAD):
            continue
        if (ty < latmin + LABEL_MIN_EDGE_PAD) or (ty > latmax - LABEL_MIN_EDGE_PAD):
            continue
        if track_lonlat.size > 0:
            dtrk = np.sqrt((track_lonlat[:, 0] - tx)**2 + (track_lonlat[:, 1] - ty)**2)
            if np.nanmin(dtrk) < LABEL_MIN_DIST_DEG:
                continue
        if used_label_anchors is not None and used_label_anchors.size > 0:
            dlbl = np.sqrt((used_label_anchors[:, 0] - tx)**2 + (used_label_anchors[:, 1] - ty)**2)
            if np.nanmin(dlbl) < LABEL_MIN_DIST_TEXT:
                continue
        return tx, ty

    return start_lon - 0.9, start_lat + 0.9

def save_density_map(count2d, lon1d, lat1d, title, out_png, cmap="gist_ncar",
                     vmin=0.0, vmax=100.0, cbar_label="Density Grid Map"):
    fig, ax, pc = make_map(title)
    add_latlon_ticks(ax, fontsize=16)

    Z = np.clip(count2d.astype(np.float32), vmin, vmax)

    pm = ax.pcolormesh(lon1d, lat1d, Z, transform=pc, cmap=cmap, shading="auto",
                       vmin=vmin, vmax=vmax, zorder=2)

    for city, (clon, clat) in CITIES.items():
        ax.plot(clon, clat, 'kD', transform=pc, zorder=10)
        ax.text(clon - 0.5, clat - 0.5, city, transform=pc,
                fontsize=14, color='k', zorder=11)

    cb_ax = fig.add_axes([0.875, 0.25, 0.02, 0.50])
    cb = plt.colorbar(pm, cax=cb_ax)
    cb.set_label(cbar_label, fontsize=16)
    cb.ax.tick_params(labelsize=16)

    fig.subplots_adjust(left=0.06, right=0.865, top=0.93, bottom=0.12)
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

# ==========================
# UPDATED: grid intensity map + optional KDE contours overlay
# ==========================
def save_grid_intensity_map(mean_int2d, lon1d, lat1d, title, out_png,
                            cmap="gist_ncar", vmin=None, vmax=None,
                            cbar_label="Grid Mean Intensity (proxy)",
                            contour_Z=None, contour_levels=None,
                            contour_color="red", contour_ls="--", contour_lw=2.2,
                            contour_label=None):
    fig, ax, pc = make_map(title)
    add_latlon_ticks(ax, fontsize=16)

    Z = mean_int2d.astype(np.float32)

    if vmin is None:
        vmin = float(np.nanpercentile(Z[np.isfinite(Z)], 2)) if np.any(np.isfinite(Z)) else 0.0
    if vmax is None:
        vmax = float(np.nanpercentile(Z[np.isfinite(Z)], 98)) if np.any(np.isfinite(Z)) else 1.0
    if not np.isfinite(vmin): vmin = 0.0
    if not np.isfinite(vmax): vmax = 1.0
    if vmax <= vmin: vmax = vmin + 1e-6

    pm = ax.pcolormesh(lon1d, lat1d, Z, transform=pc, cmap=cmap, shading="auto",
                       vmin=vmin, vmax=vmax, zorder=2)

    legend_handles = []
    # ---- overlay KDE contours (heavy_Z is already normalized 0–1) ----
    if (contour_Z is not None) and (contour_levels is not None):
        lv = _safe_levels(contour_Z, contour_levels)
        if lv is not None:
            ax.contour(lon1d, lat1d, contour_Z,
                       levels=lv,
                       colors=contour_color,
                       linestyles=contour_ls,
                       linewidths=contour_lw,
                       transform=pc, zorder=8)
            if contour_label:
                legend_handles.append(
                    Line2D([0], [0], color=contour_color, lw=contour_lw, ls=contour_ls,
                           label=contour_label)
                )

    for city, (clon, clat) in CITIES.items():
        ax.plot(clon, clat, 'kD', transform=pc, zorder=10)
        ax.text(clon - 0.5, clat - 0.5, city, transform=pc,
                fontsize=14, color='k', zorder=11)

    if legend_handles:
        ax.legend(handles=legend_handles, loc="lower left",
                  fontsize=14, frameon=True, framealpha=0.95)

    cb_ax = fig.add_axes([0.875, 0.25, 0.02, 0.50])
    cb = plt.colorbar(pm, cax=cb_ax)
    cb.set_label(cbar_label, fontsize=16)
    cb.ax.tick_params(labelsize=16)

    fig.subplots_adjust(left=0.06, right=0.865, top=0.93, bottom=0.12)
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

# ---------------- STATS HELPERS ----------------
def _scatter_with_regression(x, y, xlabel, ylabel, title, out_png):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]
    y = y[ok]

    if x.size < 3:
        print(f"[WARN] Too few valid points for regression: {title}")
        return

    fig, ax = plt.subplots(figsize=(8.2, 6.6))
    ax.scatter(x, y, s=55, edgecolor="k", linewidth=0.6, alpha=0.85)

    x_unique = np.unique(x)
    y_unique = np.unique(y)

    txt_lines = []

    # Only do regression if x varies
    if x_unique.size >= 2:
        lr = linregress(x, y)
        xx = np.linspace(np.nanmin(x), np.nanmax(x), 200)
        ax.plot(xx, lr.intercept + lr.slope * xx, linewidth=2.2)
        txt_lines.append(f"Linear fit: y = {lr.slope:.3g} x + {lr.intercept:.3g}")
    else:
        txt_lines.append("Linear fit skipped: all x values are identical")

    # Correlations also need variability
    if x_unique.size >= 2 and y_unique.size >= 2:
        r_pear, p_pear = pearsonr(x, y)
        r_spear, p_spear = spearmanr(x, y)
        txt_lines.append(f"Pearson r = {r_pear:.3f} (p={p_pear:.2g})")
        txt_lines.append(f"Spearman r = {r_spear:.3f} (p={p_spear:.2g})")
    else:
        txt_lines.append("Correlation skipped: one variable is constant")

    ax.set_xlabel(xlabel, fontsize=14)
    ax.set_ylabel(ylabel, fontsize=14)
    ax.set_title(title, fontsize=15, weight="bold")

    ax.text(
        0.02, 0.98, "\n".join(txt_lines),
        transform=ax.transAxes,
        ha="left", va="top", fontsize=12,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.9, linewidth=0.8)
    )

    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

def _histplot(x, xlabel, title, out_png, bins=18):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size < 3:
        return
    fig, ax = plt.subplots(figsize=(8.2, 6.2))
    ax.hist(x, bins=bins, edgecolor="k", linewidth=0.7, alpha=0.9)
    ax.set_xlabel(xlabel, fontsize=14)
    ax.set_ylabel("Count", fontsize=14)
    ax.set_title(title, fontsize=15, weight="bold")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

def scatter_grid_with_regression(x2d, y2d,
                                 xlabel, ylabel, title, out_png,
                                 min_points=30,
                                 stats_loc="topleft"):
    """
    Scatter plot between two gridded fields with regression + correlations

    stats_loc:
        "topleft"   → stats box at upper-left
        "lowerright"→ stats box at lower-right
    """
    x = np.asarray(x2d, dtype=np.float64).ravel()
    y = np.asarray(y2d, dtype=np.float64).ravel()

    ok = np.isfinite(x) & np.isfinite(y)
    ok = ok & (x >= 0) & (y >= 0)
    x = x[ok]
    y = y[ok]

    if x.size < min_points:
        print(f"[WARN] Not enough valid points for scatter: {out_png}")
        return

    x_unique = np.unique(x)
    y_unique = np.unique(y)

    fig, ax = plt.subplots(figsize=(8.5, 6.8))
    ax.scatter(x, y, s=28, alpha=0.55, edgecolor="k", linewidth=0.3)

    stats_lines = []

    # Correlations only if both variables vary
    if x_unique.size >= 2 and y_unique.size >= 2:
        r_p, p_p = pearsonr(x, y)
        r_s, p_s = spearmanr(x, y)
        stats_lines.append(f"Pearson r = {r_p:.3f} (p={p_p:.2g})")
        stats_lines.append(f"Spearman r = {r_s:.3f} (p={p_s:.2g})")
    else:
        stats_lines.append("Correlation skipped: one variable is constant")

    # Regression only if x varies
    if x_unique.size >= 2:
        lr = linregress(x, y)
        xx = np.linspace(np.nanmin(x), np.nanmax(x), 200)
        yy = lr.intercept + lr.slope * xx
        yy = np.clip(yy, 0, None)

        ax.plot(xx, yy, color="red", lw=2.4, label="Linear regression")
        stats_lines.append(f"Slope = {lr.slope:.3g}")
    else:
        stats_lines.append("Linear regression skipped: all x values are identical")

    ax.set_xlabel(xlabel, fontsize=15)
    ax.set_ylabel(ylabel, fontsize=15)
    ax.set_title(title, fontsize=16, weight="bold")

    stats_txt = "\n".join(stats_lines)

    if stats_loc == "lowerright":
        ax.text(0.98, 0.18, stats_txt,
                transform=ax.transAxes,
                ha="right", va="bottom",
                fontsize=10,
                bbox=dict(boxstyle="round,pad=0.22",
                          facecolor="white",
                          alpha=0.95,
                          linewidth=0.7))
    else:
        ax.text(0.02, 0.98, stats_txt,
                transform=ax.transAxes,
                ha="left", va="top",
                fontsize=11,
                bbox=dict(boxstyle="round,pad=0.25",
                          facecolor="white",
                          alpha=0.95,
                          linewidth=0.7))

    ax.grid(True, alpha=0.25)

    # Only show legend if a regression line was actually drawn
    if x_unique.size >= 2:
        ax.legend(loc="lower right", fontsize=13, frameon=True)

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)
    
def compute_mcs_lifecycle_stats(det_centroids):
    df = det_centroids.copy()
    df = df.sort_values(["track", "time"]).copy()

    out_rows = []
    for tid, g in df.groupby("track", sort=True):
        g = g.sort_values("time")
        if len(g) < 2:
            continue

        t0 = pd.to_datetime(g["time"].iloc[0])
        t1 = pd.to_datetime(g["time"].iloc[-1])
        dur_h = float((t1 - t0).total_seconds() / 3600.0) + 1.0

        lat = g["clat"].to_numpy(np.float64)
        lon = g["clon"].to_numpy(np.float64)
        times = pd.to_datetime(g["time"]).to_numpy()

        d_km = []
        dt_h = []
        for i in range(1, len(g)):
            dk = float(haversine(lat[i-1], lon[i-1], lat[i], lon[i]))
            dh = float((pd.to_datetime(times[i]) - pd.to_datetime(times[i-1])).total_seconds() / 3600.0)
            if np.isfinite(dk) and np.isfinite(dh) and dh > 0:
                d_km.append(dk)
                dt_h.append(dh)

        d_km = np.asarray(d_km, dtype=np.float64)
        dt_h = np.asarray(dt_h, dtype=np.float64)

        total_dist_km  = float(np.nansum(d_km)) if d_km.size else np.nan
        mean_speed_kmh = float(np.nanmean(d_km / dt_h)) if d_km.size else np.nan
        max_speed_kmh  = float(np.nanmax(d_km / dt_h))  if d_km.size else np.nan

        area  = g["area"].to_numpy(np.float64)
        tbmin = g["tbmin"].to_numpy(np.float64)
        inten = g["intensity"].to_numpy(np.float64)

        iA = int(np.nanargmax(area))
        iI = int(np.nanargmax(inten))
        iT = int(np.nanargmin(tbmin))

        age_h = (pd.to_datetime(g["time"]) - t0).dt.total_seconds().to_numpy(np.float64) / 3600.0
        okA = np.isfinite(age_h) & np.isfinite(area)
        if np.sum(okA) >= 3:
            lrA = linregress(age_h[okA], area[okA])
            area_slope_km2_per_h = float(lrA.slope)
        else:
            area_slope_km2_per_h = np.nan

        out_rows.append(dict(
            track=int(tid),
            start_time=t0,
            end_time=t1,
            duration_h=dur_h,
            n_hours=int(len(g)),
            max_area_km2=float(np.nanmax(area)),
            time_of_max_area=pd.to_datetime(g["time"].iloc[iA]),
            peak_intensity=float(np.nanmax(inten)),
            time_of_peak_intensity=pd.to_datetime(g["time"].iloc[iI]),
            min_tb_k=float(np.nanmin(tbmin)),
            time_of_min_tb=pd.to_datetime(g["time"].iloc[iT]),
            total_distance_km=total_dist_km,
            mean_speed_kmh=mean_speed_kmh,
            max_speed_kmh=max_speed_kmh,
            area_trend_km2_per_h=area_slope_km2_per_h
        ))

    return pd.DataFrame(out_rows)

# ---------------- lifecycle composite + peak timing + "intense probability" ----------------
def lifecycle_composite(df_centroids, n_bins=20,
                        time_col="time", track_col="track",
                        intensity_col="intensity",
                        tb_col="tbmin", area_col="area"):
    df = df_centroids.copy()
    df = df.sort_values([track_col, time_col]).copy()

    g = df.groupby(track_col)
    t0 = g[time_col].transform("min")
    t1 = g[time_col].transform("max")

    denom = (t1 - t0).dt.total_seconds().astype(np.float64)
    ok = denom > 0
    df = df.loc[ok].copy()
    denom = denom[ok]

    tau = (df[time_col] - t0[ok]).dt.total_seconds().astype(np.float64) / denom
    df["tau"] = np.clip(tau, 0.0, 1.0)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    df["tau_bin"] = np.digitize(df["tau"].to_numpy(np.float64), edges, right=True) - 1
    df = df[(df["tau_bin"] >= 0) & (df["tau_bin"] < n_bins)].copy()

    def _summ(x):
        x = np.asarray(x, dtype=np.float64)
        x = x[np.isfinite(x)]
        if x.size == 0:
            return dict(q25=np.nan, q50=np.nan, q75=np.nan, mean=np.nan, n=0)
        return dict(
            q25=float(np.nanpercentile(x, 25)),
            q50=float(np.nanpercentile(x, 50)),
            q75=float(np.nanpercentile(x, 75)),
            mean=float(np.nanmean(x)),
            n=int(x.size)
        )

    out = {}
    for vname, col in [("intensity", intensity_col), ("tbmin", tb_col), ("area", area_col)]:
        if col is None or col not in df.columns:
            continue
        stats = []
        for b in range(n_bins):
            stats.append(_summ(df.loc[df["tau_bin"] == b, col]))
        out[vname] = pd.DataFrame(stats)
    return centers, out, df

def plot_lifecycle_composites(tau, stats_dict, out_png,
                             title="MCS Lifecycle Composite (Tb/Area-derived proxies; no precip)",
                             show_intensity=True, show_tb=True, show_area=True):
    panels = []
    if show_intensity and "intensity" in stats_dict: panels.append("intensity")
    if show_tb and "tbmin" in stats_dict: panels.append("tbmin")
    if show_area and "area" in stats_dict: panels.append("area")
    if len(panels) == 0:
        return

    fig, axes = plt.subplots(len(panels), 1, figsize=(9.2, 3.2*len(panels)), sharex=True)
    if len(panels) == 1:
        axes = [axes]

    for ax, key in zip(axes, panels):
        st = stats_dict[key]
        ax.plot(tau, st["q50"].to_numpy(np.float64), linewidth=2.2)
        ax.fill_between(tau,
                        st["q25"].to_numpy(np.float64),
                        st["q75"].to_numpy(np.float64),
                        alpha=0.25, linewidth=0.0)
        ax.grid(True, alpha=0.25)

        if key == "intensity":
            ax.set_ylabel("Intensity (proxy)", fontsize=13)
            ax.set_title(title, fontsize=14, weight="bold")
        elif key == "tbmin":
            ax.set_ylabel("Tbmin (K)", fontsize=13)
        elif key == "area":
            ax.set_ylabel("Area (km$^2$)", fontsize=13)

        ax2 = ax.twinx()
        ax2.plot(tau, st["n"].to_numpy(np.float64), alpha=0.20, linewidth=1.2)
        ax2.set_ylabel("N", fontsize=10)
        ax2.tick_params(labelsize=9)

    axes[-1].set_xlabel("Normalized lifetime fraction (τ)", fontsize=13)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

def plot_peak_timing_hist(df_tau, var="intensity", track_col="track", bins=18, out_png="peak_timing_hist.png"):
    if var not in df_tau.columns:
        return

    rows = []
    for tid, g in df_tau.groupby(track_col, sort=True):
        g = g[np.isfinite(g["tau"]) & np.isfinite(g[var])]
        if len(g) < 2:
            continue
        if var == "tbmin":
            i = int(np.nanargmin(g[var].to_numpy(np.float64)))
        else:
            i = int(np.nanargmax(g[var].to_numpy(np.float64)))
        rows.append(float(g["tau"].iloc[i]))

    if len(rows) < 3:
        return

    x = np.asarray(rows, dtype=np.float64)
    fig, ax = plt.subplots(figsize=(8.0, 5.8))
    ax.hist(x, bins=bins, edgecolor="k", linewidth=0.7, alpha=0.9)
    ax.set_xlabel(f"τ at peak {var}" + (" (min)" if var == "tbmin" else " (max)"), fontsize=13)
    ax.set_ylabel("Number of MCS tracks", fontsize=13)
    ax.set_title(f"Peak Timing Across Lifecycle (τ) — {var}", fontsize=14, weight="bold")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

def plot_intense_probability_vs_tau(df_tau, intensity_col="intensity",
                                   tau_bin_col="tau_bin",
                                   n_bins=20, q=0.90,
                                   out_png="prob_intense_vs_tau.png"):
    if intensity_col not in df_tau.columns:
        return

    vals = df_tau[intensity_col].to_numpy(np.float64)
    vals = vals[np.isfinite(vals)]
    if vals.size < 10:
        return
    thr = float(np.nanquantile(vals, q))

    probs = []
    ns = []
    for b in range(n_bins):
        sub = df_tau[df_tau[tau_bin_col] == b]
        x = sub[intensity_col].to_numpy(np.float64)
        x = x[np.isfinite(x)]
        n = int(x.size)
        ns.append(n)
        if n == 0:
            probs.append(np.nan)
        else:
            probs.append(float(np.mean(x >= thr)))

    tau = (np.arange(n_bins) + 0.5) / n_bins
    probs = np.asarray(probs, dtype=np.float64)
    ns = np.asarray(ns, dtype=np.float64)

    fig, ax = plt.subplots(figsize=(8.2, 5.8))
    ax.plot(tau, probs, linewidth=2.2)
    ax.set_xlabel("Normalized lifetime fraction (τ)", fontsize=13)
    ax.set_ylabel(f"P(Intensity ≥ {q:.2f} quantile)", fontsize=13)
    ax.set_title("Probability of 'Intense' Proxy State vs Lifecycle (no precip)", fontsize=14, weight="bold")
    ax.grid(True, alpha=0.25)

    ax2 = ax.twinx()
    ax2.plot(tau, ns, alpha=0.20, linewidth=1.2)
    ax2.set_ylabel("N", fontsize=10)
    ax2.tick_params(labelsize=9)

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

# ---------------- MAIN ----------------
def main():
    if os.path.exists(DET_TMP):
        os.remove(DET_TMP)

    files = sorted(glob.glob(FILE_GLOB))
    meta = []
    for f in files:
        ts, h = parse_hour(f)
        if ts is not None:
            meta.append((ts, h, f))
    meta.sort()

    if not meta:
        print("[ERROR] No valid files found that match pattern/time.")
        return

    first_ds = xr.open_dataset(meta[0][2])
    try:
        Tb0, lat0, lon0, _ = extract_lat_lon_tb(first_ds, tb_var=TB_VAR, p_var=None)
    finally:
        first_ds.close()

    A0 = compute_area(lat0, lon0)
    last_shape = (lat0.size, lon0.size)

    ccs_count     = np.zeros(last_shape, dtype=np.int32)
    mcs_px_count  = np.zeros(last_shape, dtype=np.int32)
    mcs_int_sum   = np.zeros(last_shape, dtype=np.float32)

    prev = None
    nextID = 1
    file_lookup = {}

    # ---------------- FIRST PASS: detections + tracking ----------------
    for ts, h, f in meta:
        file_lookup[os.path.basename(f)] = f

        ds = xr.open_dataset(f)
        try:
            Tb, lat, lon, _ = extract_lat_lon_tb(ds, tb_var=TB_VAR, p_var=None)
        finally:
            ds.close()

        if (lat.size, lon.size) != last_shape or (not np.allclose(lat, lat0)) or (not np.allclose(lon, lon0)):
            raise RuntimeError(
                f"[ERROR] Grid changed between files. "
                f"First grid: {lat0.size}x{lon0.size}, this file: {lat.size}x{lon.size}. "
                f"Stop to avoid wrong density counts."
            )

        ccs_mask = np.isfinite(Tb) & (Tb < TB_CCS_THRESH)
        ccs_count += ccs_mask.astype(np.int32)

        lbl, num = label_objects(Tb)
        if A0.shape != Tb.shape or lbl.shape != Tb.shape:
            raise RuntimeError(f"Grid shape mismatch: A0{A0.shape}, Tb{Tb.shape}, lbl{lbl.shape}")

        if num == 0:
            prev = None
            del Tb, lbl
            gc.collect()
            continue

        records = []
        for oid in range(1, num+1):
            region = (lbl == oid)
            if not np.any(region):
                continue

            area_km2 = float(np.sum(A0[region]))
            if area_km2 < 200.0:
                continue

            tb_vals = Tb[region]
            if np.all(np.isnan(tb_vals)):
                continue
            tbmin = float(np.nanmin(tb_vals))

            iy, ix = np.where(region)
            cy = int(np.round(np.mean(iy))); cx = int(np.round(np.mean(ix)))
            cy = 0 if cy < 0 else (len(lat)-1 if cy >= len(lat) else cy)
            cx = 0 if cx < 0 else (len(lon)-1 if cx >= len(lon) else cx)
            clat, clon = float(lat[cy]), float(lon[cx])

            records.append((oid, area_km2, tbmin, clat, clon))

        del Tb, lbl
        gc.collect()

        if not records:
            prev = None
            continue

        cur = pd.DataFrame.from_records(records, columns=["oid","area","tbmin","clat","clon"])
        cur["area"]  = cur["area"].astype("float32")
        cur["tbmin"] = cur["tbmin"].astype("float32")
        cur["clat"]  = cur["clat"].astype("float32")
        cur["clon"]  = cur["clon"].astype("float32")

        cur["time"]  = pd.to_datetime(ts).floor("h")
        cur["hstr"]  = h
        cur["file"]  = os.path.basename(f)
        cur["track"] = -1

        if prev is not None and len(prev) > 0:
            used = set()
            cur_lat = cur["clat"].to_numpy()
            cur_lon = cur["clon"].to_numpy()
            for _, prow in prev.iterrows():
                d = haversine(prow.clat, prow.clon, cur_lat, cur_lon)
                j = int(np.argmin(d))
                if (j not in used) and (d[j] <= MAX_LINK_DIST_KM):
                    cur.iat[j, cur.columns.get_loc("track")] = int(prow.track)
                    used.add(j)

        unlinked = cur.index[cur["track"] == -1]
        if len(unlinked) > 0:
            cur.loc[unlinked, "track"] = np.arange(nextID, nextID + len(unlinked), dtype=np.int64)
            nextID += len(unlinked)

        write_chunk(cur[["oid","area","tbmin","clat","clon","time","hstr","file","track"]], DET_TMP)

        prev = cur[["clat","clon","track"]].copy()
        prev["clat"]  = prev["clat"].astype("float32")
        prev["clon"]  = prev["clon"].astype("float32")
        prev["track"] = prev["track"].astype("int64")

        del cur
        gc.collect()

    if not os.path.exists(DET_TMP):
        print("[WARN] No detections were found.")
        return

    dtypes = {
        "oid":"int32","area":"float32","tbmin":"float32","clat":"float32","clon":"float32",
        "hstr":"string","file":"string","track":"int64"
    }
    det = pd.read_csv(DET_TMP, dtype=dtypes, parse_dates=["time"])
    det["time"] = pd.to_datetime(det["time"]).dt.floor("h")
    det["time_hour"] = det["time"]

    trk = det.groupby("track", sort=True).agg(
        start_time=("time","min"),
        end_time=("time","max"),
        dur=("time_hour","nunique"),
        maxA=("area","max"),
        minTb=("tbmin","min"),
    )
    trk["mcs"] = (trk["maxA"] >= MCS_MIN_AREA_KM2) & (trk["dur"] >= MCS_MIN_DURATION_H) & (trk["minTb"] <= TB_CORE_THRESH)

    keep_ids = trk.index[trk["mcs"]].to_numpy()
    det = det[det["track"].isin(keep_ids)].copy()
    trk = trk.loc[keep_ids].copy()

    if det.empty:
        print("[INFO] No tracks met MCS criteria with current thresholds.")
        return

    # renumber tracks
    renum = {old: new for new, old in enumerate(sorted(keep_ids), start=1)}
    det["track"] = det["track"].map(renum).astype("int32")
    trk.index = trk.index.map(renum)

    # intensity proxy (per-object, per-hour)
    det["intensity"] = ((TB_CCS_THRESH - det["tbmin"]) / 50.0).clip(lower=0) + (det["area"] / (det["area"] + 40000.0))
    det["intensity"] = det["intensity"].astype("float32")

    det.to_csv(DET_OUT, index=False)
    trk.to_csv(TRK_OUT)

    det = det.sort_values(["track", "time"]).copy()
    det_centroids = det.drop_duplicates(subset=["track", "time_hour"], keep="first").copy()

    # ---------------- Build lookups for SECOND PASS ----------------
    oids_by_filehour = {}
    for (fname, th), sub in det.groupby(["file", "time_hour"]):
        oids_by_filehour[(fname, th)] = set(sub["oid"].astype(int).tolist())

    inten_by_filehour_oid = {}
    for (fname, th), sub in det.groupby(["file", "time_hour"]):
        for _, rr in sub.iterrows():
            inten_by_filehour_oid[(fname, th, int(rr["oid"]))] = float(rr["intensity"])

    # =======================
    # OVERVIEW STATS
    # =======================
    os.makedirs(FIG_DIR_OV, exist_ok=True)

    life = compute_mcs_lifecycle_stats(det_centroids)
    life_csv = os.path.join(FIG_DIR_OV, "mcs_lifecycle_summary.csv")
    life.to_csv(life_csv, index=False)

    _histplot(life["duration_h"], "Duration (hours)",
             "MCS Lifecycle: Duration Distribution",
             os.path.join(FIG_DIR_OV, "hist_duration_h.png"))

    _histplot(life["mean_speed_kmh"], "Mean Translational Speed (km h$^{-1}$)",
             "MCS Lifecycle: Mean Speed Distribution",
             os.path.join(FIG_DIR_OV, "hist_mean_speed_kmh.png"))

    _histplot(life["max_area_km2"], "Maximum Area (km$^2$)",
             "MCS Lifecycle: Maximum Area Distribution",
             os.path.join(FIG_DIR_OV, "hist_max_area_km2.png"))

    _histplot(life["min_tb_k"], "Minimum Tb (K)",
             "MCS Lifecycle: Minimum Tb Distribution",
             os.path.join(FIG_DIR_OV, "hist_min_tb_k.png"))

    _scatter_with_regression(
        life["duration_h"], life["max_area_km2"],
        "Duration (hours)", "Max Area (km$^2$)",
        "Duration vs Max Area (per tracked MCS)",
        os.path.join(FIG_DIR_OV, "scatter_duration_vs_maxarea.png")
    )

    _scatter_with_regression(
        life["duration_h"], life["peak_intensity"],
        "Duration (hours)", "Peak Intensity",
        "Duration vs Peak Intensity (per tracked MCS)",
        os.path.join(FIG_DIR_OV, "scatter_duration_vs_peakintensity.png")
    )

    _scatter_with_regression(
        life["mean_speed_kmh"], life["peak_intensity"],
        "Mean Speed (km h$^{-1}$)", "Peak Intensity",
        "Mean Speed vs Peak Intensity (per tracked MCS)",
        os.path.join(FIG_DIR_OV, "scatter_meanspeed_vs_peakintensity.png")
    )

    _scatter_with_regression(
        det_centroids["area"], det_centroids["intensity"],
        "Area (km$^2$) at time", "Intensity at time",
        "Hourly Samples: Area vs Intensity",
        os.path.join(FIG_DIR_OV, "scatter_hourly_area_vs_intensity.png")
    )

    _scatter_with_regression(
        det_centroids["tbmin"], det_centroids["intensity"],
        "Tbmin (K) at time", "Intensity at time",
        "Hourly Samples: Tbmin vs Intensity",
        os.path.join(FIG_DIR_OV, "scatter_hourly_tbmin_vs_intensity.png")
    )

    print(f"[OK] Lifecycle stats CSV: {life_csv}")
    print(f"[OK] Overview stats figures saved in: {FIG_DIR_OV}/")

    # =======================
    # Lifecycle intensification diagnostics (NO precip)
    # =======================
    tau, comp, df_tau = lifecycle_composite(det_centroids, n_bins=LIFECYCLE_NBINS)

    out_life_comp = os.path.join(FIG_DIR_OV, "lifecycle_composite_intensity_tb_area.png")
    plot_lifecycle_composites(
        tau, comp, out_life_comp,
        title="MCS Lifecycle Composite (Intensity/Tbmin/Area; no precipitation used)",
        show_intensity=True, show_tb=True, show_area=True
    )

    out_peak_inten = os.path.join(FIG_DIR_OV, "peak_timing_tau_hist_intensity.png")
    plot_peak_timing_hist(df_tau, var="intensity", bins=18, out_png=out_peak_inten)

    out_peak_tb = os.path.join(FIG_DIR_OV, "peak_timing_tau_hist_tbmin.png")
    plot_peak_timing_hist(df_tau, var="tbmin", bins=18, out_png=out_peak_tb)

    out_prob_intense = os.path.join(FIG_DIR_OV, "prob_intense_vs_tau.png")
    plot_intense_probability_vs_tau(
        df_tau,
        intensity_col="intensity",
        tau_bin_col="tau_bin",
        n_bins=LIFECYCLE_NBINS,
        q=INTENSE_Q,
        out_png=out_prob_intense
    )

    print(f"[OK] Lifecycle composite (no precip): {out_life_comp}")
    print(f"[OK] Peak timing hist (intensity):    {out_peak_inten}")
    print(f"[OK] Peak timing hist (tbmin):        {out_peak_tb}")
    print(f"[OK] P(intense|tau) (no precip):      {out_prob_intense}")

    # =======================
    # SECOND PASS: accumulate pixels + accumulate intensity in true MCS objects
    # =======================
    for ts, h, f in meta:
        base = os.path.basename(f)
        th = pd.to_datetime(ts).floor("h")
        key = (base, th)
        if key not in oids_by_filehour:
            continue

        ds = xr.open_dataset(f)
        try:
            Tb, lat, lon, _ = extract_lat_lon_tb(ds, tb_var=TB_VAR, p_var=None)
        finally:
            ds.close()

        lbl, num = label_objects(Tb)
        keep_oids = oids_by_filehour[key]

        for oid in keep_oids:
            if oid <= 0 or oid > num:
                continue

            region = (lbl == oid)
            if not np.any(region):
                continue

            mcs_px_count += region.astype(np.int32)

            inten = inten_by_filehour_oid.get((base, th, int(oid)), np.nan)
            if np.isfinite(inten):
                mcs_int_sum[region] += np.float32(inten)

        del Tb, lbl
        gc.collect()

    # KDE fields
    lon_grid = lon0.copy()
    lat_grid = lat0.copy()

    track_Z = _kde_on_grid(det_centroids["clon"].values,
                           det_centroids["clat"].values,
                           lon_grid, lat_grid,
                           bw_adjust=KDE_BW_ADJUST)

    heavy_Z = None
    if HAS_PRECIP:
        rng = np.random.default_rng(42)
        heavy_lon_all = []
        heavy_lat_all = []
        for ts, h, f in meta:
            ds = xr.open_dataset(f)
            try:
                _, lat, lon, P = extract_lat_lon_tb(ds, tb_var=TB_VAR, p_var=PRECIP_VAR)
            finally:
                ds.close()

            if P is None:
                continue

            mask = np.isfinite(P) & (P >= PRECIP_KDE_THRESH)
            if not np.any(mask):
                del P, mask
                gc.collect()
                continue

            iy, ix = np.where(mask)
            n = iy.size
            if n > MAX_PTS_PER_HOUR:
                sel = rng.choice(n, size=MAX_PTS_PER_HOUR, replace=False)
                iy = iy[sel]; ix = ix[sel]

            heavy_lat_all.append(lat[iy].astype(np.float64))
            heavy_lon_all.append(lon[ix].astype(np.float64))

            del P, mask
            gc.collect()

        if len(heavy_lon_all) > 0:
            heavy_lon_all = np.concatenate(heavy_lon_all)
            heavy_lat_all = np.concatenate(heavy_lat_all)
            heavy_Z = _kde_on_grid(heavy_lon_all, heavy_lat_all, lon_grid, lat_grid, bw_adjust=KDE_BW_ADJUST)

    # OVERVIEW map (centroid intensity + KDE contours)
    os.makedirs(FIG_DIR_OV, exist_ok=True)
    fig, ax, pc = make_map("MCS Intensity with Track KDE and Heavy-Precip KDE")
    add_latlon_ticks(ax, fontsize=16)

    I = det_centroids["intensity"].to_numpy(np.float32)

    # ---- NEW: safe scaling (avoid divide-by-zero if all values identical) ----
    Imin = float(np.nanmin(I))
    Imax = float(np.nanmax(I))
    den = (Imax - Imin)
    if (not np.isfinite(den)) or den <= 0:
        I_scaled = np.full_like(I, 120.0, dtype=np.float32)
    else:
        I_scaled = 40.0 + 260.0 * ((I - Imin) / den)

    sc = ax.scatter(det_centroids["clon"], det_centroids["clat"], transform=pc,
                    s=I_scaled, c=I, cmap="Blues",
                    edgecolor="black", linewidth=0.5, alpha=0.85, zorder=30)

    lv_track = _safe_levels(track_Z, KDE_LEVELS)
    if (track_Z is not None) and (lv_track is not None):
        ax.contour(lon_grid, lat_grid, track_Z,
                   levels=lv_track, colors="black", linewidths=2.4,
                   transform=pc, zorder=40)

    lv_heavy = _safe_levels(heavy_Z, KDE_LEVELS)
    if (heavy_Z is not None) and (lv_heavy is not None):
        ax.contour(lon_grid, lat_grid, heavy_Z,
                   levels=lv_heavy, colors="red", linewidths=2.2, linestyles="--",
                   transform=pc, zorder=41)

    for city, (clon, clat) in CITIES.items():
        ax.plot(clon, clat, 'kD', transform=pc, zorder=60)
        ax.text(clon - 0.5, clat - 0.5, city, transform=pc,
                fontsize=14, color='k', zorder=61)

    cb_ax = fig.add_axes([0.80, 0.35, 0.02, 0.50])
    cb = plt.colorbar(sc, cax=cb_ax)
    cb.set_label("MCS Intensity Index", fontsize=16)
    cb.ax.tick_params(labelsize=16)

    legend_handles = [
        Line2D([0], [0], color="black", lw=2.4, label="MCS Track KDE (normalized 0–1)"),
        Line2D([0], [0], color="red",   lw=2.2, ls="--",
               label=rf"Hourly Precip KDE (P ≥ {PRECIP_KDE_THRESH:g} mm h$^{{-1}}$, normalized 0–1)")
    ]
    ax.legend(handles=legend_handles,
              loc="lower center", bbox_to_anchor=(0.5, -0.17),
              ncol=2, fontsize=14, frameon=True, framealpha=0.95)

    fig.subplots_adjust(left=0.06, right=0.865, top=0.93, bottom=0.25)
    out_overview = os.path.join(FIG_DIR_OV, "mcs_intensity_with_kde_contours_labeled.png")
    plt.savefig(out_overview, dpi=300, bbox_inches="tight")
    plt.close(fig)

    # =======================
    # DENSITY MAPS
    # =======================
    out_mcs_px = os.path.join(FIG_DIR_OV, "density_ALLpixels_in_TRUE_MCS_objects_counts.png")
    save_density_map(
        mcs_px_count, lon0, lat0,
        "Density: All Grid Points Inside TRUE MCS Tracked Objects",
        out_mcs_px,
        cmap="gist_ncar",
        vmin=0.0, vmax=100.0,
        cbar_label="Density Grid Map"
    )

    # ---- FIX: define out_ccs (previously printed but never created) ----
    out_ccs = os.path.join(FIG_DIR_OV, "density_CCS_pixels_TbLT241K_counts.png")
    save_density_map(
        ccs_count, lon0, lat0,
        f"Density: Cloud Shield Pixels (Tb < {TB_CCS_THRESH:g} K)",
        out_ccs,
        cmap="gist_ncar",
        vmin=0.0, vmax=100.0,
        cbar_label="Density Grid Map (# hours)"
    )

    # grid-mean intensity map (derived from the same TRUE-object pixels)
    mcs_int_mean = np.full_like(mcs_int_sum, np.nan, dtype=np.float32)
    ok = mcs_px_count > 0
    mcs_int_mean[ok] = mcs_int_sum[ok] / mcs_px_count[ok].astype(np.float32)

    out_mcs_int_grid = os.path.join(FIG_DIR_OV, "intensity_grid_TRUE_MCS_objects_mean.png")
    save_grid_intensity_map(
        mcs_int_mean, lon0, lat0,
        "Grid Mean Intensity: TRUE MCS Object Pixels (time-averaged proxy)",
        out_mcs_int_grid,
        cmap="gist_ncar",
        vmin=None, vmax=None,
        cbar_label="Grid Mean Intensity (proxy)"
    )

    # ==========================================================
    # MULTIPLIED MAP = density * mean_intensity (== mcs_int_sum)
    # ==========================================================
    mcs_mult = mcs_px_count.astype(np.float32) * mcs_int_mean  # == cumulative intensity sum

    out_mcs_mult = os.path.join(FIG_DIR_OV, "mult_density_times_meanIntensity_TRUE_MCS_objects.png")
    save_grid_intensity_map(
        mcs_mult, lon0, lat0,
        "TRUE MCS Object Pixels: Density × Grid-Mean Intensity (== Cumulative Intensity)",
        out_mcs_mult,
        cmap="gist_ncar",
        vmin=None, vmax=None,
        cbar_label="Cumulative Intensity (proxy)"
    )

    # ==========================================================
    # COMPANION 1: Normalized cumulative intensity (0–1)
    # + Extreme precip KDE contours
    # ==========================================================
    mcs_mult_norm = np.full_like(mcs_mult, np.nan, dtype=np.float32)
    finite = np.isfinite(mcs_mult)
    if np.any(finite):
        mmin = float(np.nanmin(mcs_mult[finite]))
        mmax = float(np.nanmax(mcs_mult[finite]))
        mcs_mult_norm[finite] = (mcs_mult[finite] - mmin) / (mmax - mmin + 1e-6)

    out_mcs_mult_norm = os.path.join(
        FIG_DIR_OV,
        "mult_density_times_meanIntensity_TRUE_MCS_objects_NORMALIZED.png"
    )
    save_grid_intensity_map(
        mcs_mult_norm, lon0, lat0, "Convoluted MCS track density and mean intensity",
        out_mcs_mult_norm,
        cmap="gist_ncar",
        vmin=0.0, vmax=1.0,
        cbar_label="Normalized MCS Cumulative Intensity (0–1; density × mean intensity)",
        contour_Z=heavy_Z,
        contour_levels=KDE_LEVELS,
        contour_color="black",
        contour_ls="--",
        contour_lw=2.4,
        contour_label=rf"Extreme precip KDE (P ≥ {PRECIP_KDE_THRESH:g} mm h$^{{-1}}$, normalized 0–1)"
    )
    
    out_scatter_density_vs_meanI = os.path.join(
        FIG_DIR_OV,
        "scatter_density_vs_meanIntensity_TRUE_MCS_objects.png"
    )
    
    scatter_grid_with_regression(
        mcs_px_count,
        mcs_int_mean,
        xlabel="MCS Track Density (# hours)",
        ylabel="Mean MCS Intensity (proxy)",
        title="MCS Track Density vs Mean Intensity",
        out_png=out_scatter_density_vs_meanI
    )
    
    if heavy_Z is not None:
        out_scatter_mult_vs_precip = os.path.join(
            FIG_DIR_OV,
            "scatter_normCumulativeIntensity_vs_extremePrecipKDE.png"
        )
    
        scatter_grid_with_regression(
            mcs_mult_norm,
            heavy_Z,
            xlabel="Normalized MCS Cumulative Intensity (0–1)",
            ylabel=f"Extreme Precip KDE (P ≥ {PRECIP_KDE_THRESH:g} mm h$^{{-1}}$, 0–1)",
            title="MCS Cumulative Intensity vs Extreme Precipitation Hotspots",
            out_png=out_scatter_mult_vs_precip
        )

    if heavy_Z is not None:

        # -----------------------------------------
        # 1) Density vs Extreme Precipitation KDE
        # -----------------------------------------
        out_scatter_density_vs_precip = os.path.join(
            FIG_DIR_OV,
            "scatter_density_vs_extremePrecipKDE.png"
        )

        scatter_grid_with_regression(
            mcs_px_count,
            heavy_Z,
            xlabel="MCS Track Density (# hours)",
            ylabel=f"Extreme Precip KDE (P ≥ {PRECIP_KDE_THRESH:g} mm h$^{{-1}}$, 0–1)",
            title="MCS Track Density vs Extreme Precipitation Hotspots",
            out_png=out_scatter_density_vs_precip,
            stats_loc="topleft"
        )


        # -----------------------------------------
        # 2) Mean Intensity vs Extreme Precipitation KDE
        # -----------------------------------------
        out_scatter_meanI_vs_precip = os.path.join(
            FIG_DIR_OV,
            "scatter_meanIntensity_vs_extremePrecipKDE.png"
        )

        scatter_grid_with_regression(
            mcs_int_mean,
            heavy_Z,
            xlabel="Mean MCS Intensity (proxy)",
            ylabel=f"Extreme Precip KDE (P ≥ {PRECIP_KDE_THRESH:g} mm h$^{{-1}}$, 0–1)",
            title="Mean MCS Intensity vs Extreme Precipitation Hotspots",
            out_png=out_scatter_meanI_vs_precip,
            stats_loc="topleft"
        )

    # =======================
    # SNAPSHOTS
    # =======================
    os.makedirs(FIG_DIR_SNP, exist_ok=True)

    precip_cmap = ListedColormap(PRECIP_COLOURS)
    precip_norm = BoundaryNorm(PRECIP_LEVELS, precip_cmap.N, clip=True)
    contour_lvls = [l for l in PRECIP_LEVELS if l % 2 == 0 and l > 0]

    start_times = det.groupby("track")["time"].min()
    track_label = {tid: f"MCS {tid:02d} ({start_times.loc[tid].strftime('%Y%m%d-%H')})" for tid in start_times.index}

    for tid, track_df in det.groupby("track", sort=True):
        track_df = track_df.sort_values("time").copy()
        track_df["step"] = np.arange(1, len(track_df) + 1)
        mcs_name = track_label[tid]

        start_row  = track_df.iloc[0]
        start_lon  = float(start_row["clon"])
        start_lat  = float(start_row["clat"])
        start_text = f"MCS {int(tid):02d} Start"

        track_pts = track_df[["clon","clat"]].to_numpy(np.float64)
        city_pts  = np.array([[v[0], v[1]] for v in CITIES.values()], dtype=np.float64) if len(CITIES) else np.empty((0,2))
        avoid_pts = np.vstack([track_pts, city_pts]) if (track_pts.size or city_pts.size) else np.empty((0,2))

        used_label_anchors = []

        for _, r in track_df.iterrows():
            nc_path = file_lookup.get(r["file"])
            if nc_path is None:
                continue

            ds = xr.open_dataset(nc_path)
            try:
                Tb, lat, lon, P = extract_lat_lon_tb(ds, tb_var=TB_VAR, p_var=PRECIP_VAR if HAS_PRECIP else None)
            finally:
                ds.close()

            mask_ccs = (Tb < TB_CCS_THRESH)
            Tb_ccs = np.where(mask_ccs, Tb, np.nan)

            time_label = pd.to_datetime(r["time"]).strftime("%Y-%m-%d %H UTC")
            title = f"{mcs_name} | {time_label} | Step {int(r['step']):02d}"
            fig, ax, pc = make_map(title)
            add_latlon_ticks(ax, fontsize=16)

            fig.subplots_adjust(left=0.06, right=0.94, top=0.93, bottom=0.32)

            # precip
            m_prec = None
            if P is not None:
                m_prec = ax.pcolormesh(lon, lat, P, transform=pc,
                                       cmap=precip_cmap, norm=precip_norm,
                                       shading="auto", alpha=1.0, zorder=3)

            # CCS Tb overlay
            m_obj = ax.pcolormesh(lon, lat, Tb_ccs, transform=pc, cmap="Blues",
                                  vmin=190, vmax=240, shading="auto", alpha=0.35, zorder=5)

            # object boundary ONLY (no fill)
            if SHOW_OBJECT_PIXELS_IN_SNAPSHOTS:
                lbl, num = label_objects(Tb)
                oid_now = int(r["oid"])
            
                if 1 <= oid_now <= num:
                    obj_mask = (lbl == oid_now)
            
                    # Draw ONLY the boundary in black
                    ax.contour(
                        lon, lat,
                        obj_mask.astype(np.int8),
                        levels=[0.5],
                        colors='blue',       # <-- blue border
                        linewidths=1.5,       # adjust thickness if needed
                        transform=pc,
                        zorder=4
                    )
            
                del lbl
                gc.collect()

            # track path/markers
            hist = track_df[track_df["time"] <= r["time"]]
            ax.plot(hist["clon"], hist["clat"], transform=pc, color="black",
                    linestyle=":", linewidth=2.0, zorder=7)
            ax.scatter(hist["clon"], hist["clat"], transform=pc, s=20, color="black",
                       edgecolor="black", linewidth=0.5, zorder=8)

            for _, pt in hist.iterrows():
                ax.text(pt["clon"] + 0.08, pt["clat"] + 0.08, f"{int(pt['step']):02d}",
                        fontsize=9, weight="bold", color="black", transform=pc, zorder=9)

            ax.scatter(r["clon"], r["clat"], transform=pc, s=40, color="yellow",
                       edgecolor="black", linewidth=1.0, zorder=10)

            # start marker + label
            if PLOT_START_ON_ALL_STEPS or int(r["step"]) == 1:
                ax.scatter(start_lon, start_lat, transform=pc, s=160,
                           marker="*", color="lime",
                           edgecolor="black", linewidth=1.0, zorder=9)

                tx, ty = choose_nonoverlap_label_anchor(
                    start_lon, start_lat,
                    track_lonlat=avoid_pts,
                    used_label_anchors=used_label_anchors,
                    extent=MAP_EXTENT
                )
                used_label_anchors.append([tx, ty])

                ax.annotate(
                    start_text,
                    xy=(start_lon, start_lat), xycoords=pc._as_mpl_transform(ax),
                    xytext=(tx, ty), textcoords=pc._as_mpl_transform(ax),
                    fontsize=START_TEXT_FS, weight="bold", color=START_TEXT_COLOR,
                    ha="left", va="bottom",
                    arrowprops=dict(arrowstyle="->", lw=START_ARROW_LW, color=START_TEXT_COLOR),
                    zorder=13
                )

            # cities
            for city, (clon, clat) in CITIES.items():
                ax.plot(clon, clat, 'kD', transform=pc, zorder=11)
                ax.text(clon - 0.5, clat - 0.5, city, transform=pc,
                        fontsize=12, color='k', zorder=12)

            # COLORBARS
            cax_obj = fig.add_axes([0.83, 0.36, 0.02, 0.50])
            cb_obj = plt.colorbar(m_obj, cax=cax_obj)
            cb_obj.set_label("MCS Tb (K)", fontsize=16)
            cb_obj.ax.tick_params(labelsize=16, length=0)
            cb_obj.outline.set_linewidth(0)

            if m_prec is not None:
                cax_p = fig.add_axes([0.12, 0.25, 0.76, 0.028])
                cb_p = plt.colorbar(m_prec, cax=cax_p, orientation='horizontal',
                                    ticks=PRECIP_LEVELS[::2], extend='max')
                cb_p.set_label(f"Precipitation ({PRECIP_UNITS})", fontsize=16)
                cb_p.ax.tick_params(labelsize=16)

            obj_handle   = Line2D([0], [0], color=OBJ_EDGE_COLOR, lw=OBJ_EDGE_LW,
                                  label="MCS Object Grids (this hour)")
            start_handle = Line2D([0], [0], marker='*', color='w',
                                  markerfacecolor='lime', markeredgecolor='k',
                                  markersize=14, linestyle='None', label="MCS Start")
            path_handle  = Line2D([0], [0], color="black", lw=2.0, ls=":", label="MCS Path")
            cur_handle   = Line2D([0], [0], marker='o', color='w',
                                  markerfacecolor='yellow', markeredgecolor='k',
                                  markersize=10, linestyle='None', label="Current Centroid")

            fig.legend(
                handles=[path_handle, obj_handle, cur_handle, start_handle],
                loc="lower center",
                bbox_to_anchor=(0.5, 0.080),
                ncol=2,
                fontsize=15,
                frameon=True,
                framealpha=0.95,
                columnspacing=1.8,
                handlelength=2.2,
                borderaxespad=0.2
            )

            safe_name = mcs_name.replace(" ", "_").replace("(", "").replace(")", "")
            out_png = os.path.join(FIG_DIR_SNP, f"{safe_name}_step_{int(r['step']):02d}.png")
            plt.savefig(out_png, dpi=300, bbox_inches="tight")
            plt.close(fig)

            del Tb, Tb_ccs, P
            gc.collect()

    print(f"[OK] Overview saved: {out_overview}")
    print(f"[OK] Pixel density (TRUE MCS objects): {out_mcs_px}")
    print(f"[OK] Grid mean intensity (TRUE MCS objects): {out_mcs_int_grid}")
    print(f"[OK] Multiplied map (cumulative intensity): {out_mcs_mult}")
    print(f"[OK] Normalized cumulative map (with extreme precip KDE contours): {out_mcs_mult_norm}")
    print(f"[OK] Pixel density (Tb < {TB_CCS_THRESH:g}K): {out_ccs}")
    print(f"[OK] Detections CSV: {DET_OUT}")
    print(f"[OK] Tracks CSV: {TRK_OUT}")
    print("det_centroids area unique count:", det_centroids["area"].nunique())
    print("det_centroids area min/max:", det_centroids["area"].min(), det_centroids["area"].max())
    print("det_centroids tbmin min/max:", det_centroids["tbmin"].min(), det_centroids["tbmin"].max())
    print("det_centroids intensity min/max:", det_centroids["intensity"].min(), det_centroids["intensity"].max())

if __name__ == "__main__":
    main()

[WARN] No detections were found.


### MPAS regridding to MERGIR

In [4]:
import xarray as xr
import numpy as np
from scipy.interpolate import griddata
import glob
from pathlib import Path

# ============================================================
# Reference MERGIR grid
# ============================================================
reference_file = '/glade/work/sshohan/MERGIR_jun_jul_2025/subset_Tb/subset_merg_2025073123_4km-pixel.nc4'

with xr.open_dataset(reference_file) as ds_ref:
    lat_new = ds_ref['lat'].values   # expected 1D
    lon_new = ds_ref['lon'].values   # expected 1D

    lon_new_grid, lat_new_grid = np.meshgrid(lon_new, lat_new)

    # copy reference coord attrs if available
    lat_ref_attrs = ds_ref['lat'].attrs if 'lat' in ds_ref else {}
    lon_ref_attrs = ds_ref['lon'].attrs if 'lon' in ds_ref else {}

# ============================================================
# Input/output files
# ============================================================
file_list = sorted(glob.glob('/glade/work/sshohan/MPAS_MCS_Tracking/hourly_subset/*.nc'))

output_dir = Path('/glade/work/sshohan/MPAS_MCS_Tracking/hourly_subset/regridded_to_MERGIR')
output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================
# Fill value
# ============================================================
FILL_VALUE = -9999.0

# ============================================================
# Regridding helper
# ============================================================
def regrid_2d_field(lat2d, lon2d, field2d, lat_target_grid, lon_target_grid,
                    method='linear', fill_value=FILL_VALUE, src_fill_value=None):
    """
    Regrid one 2D field from curvilinear MPAS grid to regular MERGIR grid.
    """
    field = np.array(field2d, dtype=np.float64)

    if src_fill_value is not None:
        field = np.where(field == src_fill_value, np.nan, field)

    valid = np.isfinite(field) & np.isfinite(lat2d) & np.isfinite(lon2d)

    if valid.sum() == 0:
        return np.full(lat_target_grid.shape, fill_value, dtype=np.float32)

    points = np.column_stack((lon2d[valid], lat2d[valid]))
    values = field[valid]

    out = griddata(
        points,
        values,
        (lon_target_grid, lat_target_grid),
        method=method,
        fill_value=np.nan
    )

    # Optional nearest fill for holes left by linear interpolation
    if method == 'linear' and np.isnan(out).any():
        out_nearest = griddata(
            points,
            values,
            (lon_target_grid, lat_target_grid),
            method='nearest',
            fill_value=np.nan
        )
        out = np.where(np.isnan(out), out_nearest, out)

    out = np.where(np.isfinite(out), out, fill_value).astype(np.float32)
    return out

# ============================================================
# Loop through all files
# ============================================================
for filepath in file_list:
    print(f'Processing: {filepath}')

    with xr.open_dataset(filepath) as ds:
        # --------------------------------------------------------
        # Read source grid (2D)
        # --------------------------------------------------------
        lat2d = ds['lat'].values   # shape: (rlat, rlon)
        lon2d = ds['lon'].values   # shape: (rlat, rlon)

        # --------------------------------------------------------
        # Read time
        # --------------------------------------------------------
        if 'time' in ds:
            time_vals = ds['time'].values
            time_attrs = ds['time'].attrs
        else:
            time_vals = np.array([0.0])
            time_attrs = {'long_name': 'time'}

        nt = len(time_vals)

        # --------------------------------------------------------
        # Read source variables
        # --------------------------------------------------------
        prec = ds['prec_total'].values if 'prec_total' in ds.variables else None
        ctt  = ds['ctt'].values if 'ctt' in ds.variables else None

        prec_fill_src = ds['prec_total'].attrs.get('_FillValue', None) if 'prec_total' in ds.variables else None
        ctt_fill_src  = ds['ctt'].attrs.get('_FillValue', None) if 'ctt' in ds.variables else None

        # --------------------------------------------------------
        # Allocate output arrays
        # --------------------------------------------------------
        prec_new = None
        ctt_new = None

        if prec is not None:
            prec_new = np.full((nt, len(lat_new), len(lon_new)), FILL_VALUE, dtype=np.float32)

        if ctt is not None:
            ctt_new = np.full((nt, len(lat_new), len(lon_new)), FILL_VALUE, dtype=np.float32)

        # --------------------------------------------------------
        # Regrid over time
        # --------------------------------------------------------
        for t in range(nt):
            if prec is not None:
                prec_new[t, :, :] = regrid_2d_field(
                    lat2d, lon2d, prec[t, :, :],
                    lat_new_grid, lon_new_grid,
                    method='linear',   # good default for precip field
                    fill_value=FILL_VALUE,
                    src_fill_value=prec_fill_src
                )

            if ctt is not None:
                ctt_new[t, :, :] = regrid_2d_field(
                    lat2d, lon2d, ctt[t, :, :],
                    lat_new_grid, lon_new_grid,
                    method='nearest',  # often better for preserving sharp cloud edges
                    fill_value=FILL_VALUE,
                    src_fill_value=ctt_fill_src
                )

        # --------------------------------------------------------
        # Build output dataset in MERGIR-like format
        # --------------------------------------------------------
        data_vars = {}

        if prec_new is not None:
            data_vars['prec_total'] = (('time', 'lat', 'lon'), prec_new)

        if ctt_new is not None:
            data_vars['ctt'] = (('time', 'lat', 'lon'), ctt_new)

        ds_new = xr.Dataset(
            data_vars=data_vars,
            coords={
                'time': ('time', time_vals),
                'lat': ('lat', lat_new),
                'lon': ('lon', lon_new),
            }
        )

        # --------------------------------------------------------
        # Coordinate attributes
        # --------------------------------------------------------
        ds_new['time'].attrs = time_attrs
        ds_new['lat'].attrs = lat_ref_attrs if lat_ref_attrs else {
            'units': 'degrees_north',
            'long_name': 'latitude'
        }
        ds_new['lon'].attrs = lon_ref_attrs if lon_ref_attrs else {
            'units': 'degrees_east',
            'long_name': 'longitude'
        }

        # --------------------------------------------------------
        # Variable attributes
        # --------------------------------------------------------
        if 'prec_total' in ds_new:
            ds_new['prec_total'].attrs = {
                'units': ds['prec_total'].attrs.get('units', 'mm hr-1'),
                'long_name': ds['prec_total'].attrs.get('long_name', 'hourly total precipitation (nc + c)'),
                'regrid': 'scipy.griddata linear MPAS→MERGIR'
            }
            ds_new['prec_total'].encoding['_FillValue'] = FILL_VALUE
            ds_new['prec_total'].encoding['dtype'] = 'float32'
            ds_new['prec_total'].encoding['zlib'] = True
            ds_new['prec_total'].encoding['complevel'] = 1

        if 'ctt' in ds_new:
            ds_new['ctt'].attrs = {
                'units': ds['ctt'].attrs.get('units', 'K'),
                'long_name': ds['ctt'].attrs.get('long_name', 'cloud top temperature'),
                'regrid': 'scipy.griddata nearest MPAS→MERGIR'
            }
            ds_new['ctt'].encoding['_FillValue'] = FILL_VALUE
            ds_new['ctt'].encoding['dtype'] = 'float32'
            ds_new['ctt'].encoding['zlib'] = True
            ds_new['ctt'].encoding['complevel'] = 1

        ds_new['time'].encoding['dtype'] = 'float64'

        # --------------------------------------------------------
        # Global attributes
        # --------------------------------------------------------
        ds_new.attrs = {
            'title': 'MPAS hourly fields regridded to MERGIR grid',
            'source_mesh': ds.attrs.get('source_mesh', 'MPAS'),
            'reference_grid': reference_file,
            'created_from': str(filepath),
            'history': f"Regridded to MERGIR grid from {Path(filepath).name}",
        }

        # --------------------------------------------------------
        # Write output
        # --------------------------------------------------------
        input_name = Path(filepath).name
        output_filename = output_dir / f"regridded_{input_name}"

        ds_new.to_netcdf(output_filename)

    print(f"Saved: {output_filename}")

Processing: /glade/work/sshohan/MPAS_MCS_Tracking/hourly_subset/MPAS_hourly_Texas_subset_20150504_00.nc
Saved: /glade/work/sshohan/MPAS_MCS_Tracking/hourly_subset/regridded_to_MERGIR/regridded_MPAS_hourly_Texas_subset_20150504_00.nc
Processing: /glade/work/sshohan/MPAS_MCS_Tracking/hourly_subset/MPAS_hourly_Texas_subset_20150504_01.nc
Saved: /glade/work/sshohan/MPAS_MCS_Tracking/hourly_subset/regridded_to_MERGIR/regridded_MPAS_hourly_Texas_subset_20150504_01.nc
Processing: /glade/work/sshohan/MPAS_MCS_Tracking/hourly_subset/MPAS_hourly_Texas_subset_20150504_02.nc
Saved: /glade/work/sshohan/MPAS_MCS_Tracking/hourly_subset/regridded_to_MERGIR/regridded_MPAS_hourly_Texas_subset_20150504_02.nc
Processing: /glade/work/sshohan/MPAS_MCS_Tracking/hourly_subset/MPAS_hourly_Texas_subset_20150504_03.nc
Saved: /glade/work/sshohan/MPAS_MCS_Tracking/hourly_subset/regridded_to_MERGIR/regridded_MPAS_hourly_Texas_subset_20150504_03.nc
Processing: /glade/work/sshohan/MPAS_MCS_Tracking/hourly_subset/MPAS

### Animation Creation

In [ ]:
#!/usr/bin/env python3
"""
Make a GIF animation from PNG frames using only Pillow.
- No imageio, no ffmpeg, no ImageMagick needed.
- Natural sorting: ..._step_2.png < ..._step_10.png
- Optional resize to a fixed W x H (keeps aspect via letterbox).
"""

import glob
import re
from pathlib import Path
from PIL import Image

# ====================== CONFIG ======================
# Glob pattern to find your frames (edit this)
PATTERN   = "/glade/u/home/sshohan/MCS_Tracking/figs_snapshots/MCS_*.png"   # e.g., "*.png" or "frames/*.png"
OUT_GIF   = "animation.gif"                   # output filename
FPS       = 1                                # frames per second
TARGET_SZ = None                              # None = use first frame size; or (W, H) e.g., (1280, 720)
LOOP      = 0                                 # 0 = loop forever; N = loop N times
OPTIMIZE  = True                              # Pillow GIF optimization (smaller file)
# ====================================================

def natural_key(p: Path):
    """Sort path names like a human (file2 < file10)."""
    parts = re.split(r'(\d+)', p.name)
    return [int(s) if s.isdigit() else s.lower() for s in parts]

def load_and_fit(path: Path, target_size=None):
    """
    Load an image and (optionally) fit to target_size with aspect preserved
    by letterboxing onto a black canvas.
    """
    im = Image.open(path).convert("RGB")
    if target_size is None:
        return im
    tw, th = target_size
    iw, ih = im.size
    scale = min(tw / iw, th / ih)
    nw, nh = max(1, int(iw * scale)), max(1, int(ih * scale))
    im_resized = im.resize((nw, nh), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (tw, th), (0, 0, 0))
    ox, oy = (tw - nw) // 2, (th - nh) // 2
    canvas.paste(im_resized, (ox, oy))
    return canvas

def main():
    paths = sorted([Path(p) for p in glob.glob(PATTERN)], key=natural_key)
    if not paths:
        raise SystemExit(f"No files matched pattern: {PATTERN}")

    print(f"Found {len(paths)} frames.")
    print(f"Example: {paths[0].name} … {paths[-1].name}")

    # Determine target size
    if TARGET_SZ is None:
        with Image.open(paths[0]) as im0:
            target_size = im0.size
    else:
        target_size = TARGET_SZ

    print(f"Target size: {target_size[0]}x{target_size[1]}; FPS: {FPS}")

    # Load all frames (and fit if needed)
    frames = [load_and_fit(p, target_size) for p in paths]

    # Duration per frame in milliseconds for GIF
    duration_ms = max(1, int(1000 / max(1, FPS)))

    # Save GIF with Pillow
    # Note: disposal=2 helps reduce artifacts with letterboxing/backgrounds.
    first, rest = frames[0], frames[1:]
    first.save(
        OUT_GIF,
        save_all=True,
        append_images=rest,
        duration=duration_ms,
        loop=LOOP,
        optimize=OPTIMIZE,
        disposal=2,
        quality=100,
    )
    print(f"Done: {OUT_GIF}")

if __name__ == "__main__":
    main()